# 🎓 Graduation Project — Phase 1
## A Comparative Study of Fraud Detection Models under Class Imbalance
### Dataset: Kaggle Credit Card Fraud Detection

---

**Author:** Nouha Laataoui | **Supervisor:** Nadjib Brahimi | **Institution:** Rennes School of Business

This notebook implements a rigorous comparative evaluation of four fraud detection models under extreme class imbalance. It covers:
- Exploratory Data Analysis (EDA)
- Baseline benchmark with 95% bootstrap confidence intervals
- Robustness analysis across 5 random seeds
- Sensitivity analysis: imbalance ratio & dataset size
- Feature importance consensus across tree-based models
- Probability calibration (ECE + Brier score)
- Statistical significance tests (paired t-test + Wilcoxon)

---

## ⚙️ 0 — Configuration & Setup

In [1]:
import sys
print(sys.executable)

import numpy as np
import shap
import lightgbm

print("numpy:", np.__version__)
print("shap:", shap.__version__)
print("lightgbm:", lightgbm.__version__)

c:\Users\Nouha Laataoui\anaconda3\envs\fraud-shap\python.exe
numpy: 1.26.4
shap: 0.46.0
lightgbm: 4.6.0


In [2]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# Graduation Project – Phase 1 (Corrected)
# Dataset 1: Kaggle Credit Card Fraud (creditcard.csv)

# Run:
#   Option A – auto-detect: place creditcard.csv in ./data/ next to this notebook
#   Option B – env var:     DATA_PATH=/path/to/creditcard.csv
#   Option C – explicit:    set DATA_PATH variable in CONFIG section below
# ============================================================

import os, time, json, warnings
from pathlib import Path
from collections import Counter
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import ttest_rel, wilcoxon
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    precision_recall_curve, auc, average_precision_score,
    precision_recall_fscore_support, roc_auc_score,
    roc_curve, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# ─── optional libs ───────────────────────────────────────────────
XGB_AVAILABLE = LGBM_AVAILABLE = False
try:
    import xgboost as xgb;  XGB_AVAILABLE  = True
except Exception: print("WARNING: xgboost not available.")
try:
    import lightgbm as lgb; LGBM_AVAILABLE = True
except Exception: print("WARNING: lightgbm not available.")

# ============================================================
# 0) CONFIG
# ============================================================
SEED        = 42
TEST_SIZE   = 0.15
VAL_SIZE    = 0.15
TRAIN_SIZE  = 1.0 - TEST_SIZE - VAL_SIZE   # 0.70
N_REPEATS   = 5          # FIX: increased from 3 → 5 for more stable estimates
N_BOOTSTRAP = 300

TARGET_COL   = "Class"
FRAUD_LABEL  = 1
NORMAL_LABEL = 0

# FIX: IMB_RATIOS now includes 0.001 to match Phase 2 comparison range
IMB_RATIOS  = [0.001, 0.005, 0.01, 0.05]
SIZE_FRACS  = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
RECALL_GRID = np.linspace(0, 1, 500)

# ── Paths ────────────────────────────────────────────────────
# Update DATA_PATH if you move the dataset.
NOTEBOOK_DIR = Path(os.getcwd())
DATA_PATH = Path(r"C:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Datasets\Credit Card Fraud Detection (Kaggle).csv")
OUT_DIR   = NOTEBOOK_DIR / "output phase 1"
FIG_DIR      = OUT_DIR / "figures"
RES_DIR      = OUT_DIR / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)

print(f"DATA_PATH : {DATA_PATH}")
print(f"OUTPUT    : {OUT_DIR}")
print(f"N_REPEATS : {N_REPEATS}")


# ── Colorblind-safe palette (Wong 2011) ──────────────────────────
MODEL_NAMES = ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM"]
COLORS  = {
    "Logistic Regression": "#0072B2",   # blue
    "Random Forest":       "#009E73",   # green
    "XGBoost":             "#E69F00",   # amber
    "LightGBM":            "#D55E00",   # vermillion
}
MARKERS = {"Logistic Regression": "o", "Random Forest": "s",
           "XGBoost": "^",              "LightGBM": "D"}

# ── Professional figure style ────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":          130,
    "figure.facecolor":    "white",
    "axes.facecolor":      "#F8F8F8",
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           True,
    "grid.alpha":          0.3,
    "grid.color":          "#CCCCCC",
    "grid.linestyle":      "--",
    "font.family":         "DejaVu Sans",
    "font.size":           12,
    "axes.titlesize":      14,
    "axes.titleweight":    "bold",
    "axes.labelsize":      12,
    "xtick.labelsize":     10,
    "ytick.labelsize":     10,
    "legend.fontsize":     10,
    "legend.framealpha":   0.9,
    "lines.linewidth":     2.2,
    "patch.linewidth":     0.8,
})

# ============================================================
# HELPERS
# ============================================================
def save_fig(filename: str, tight: bool = True) -> None:
    if tight: plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=200, bbox_inches="tight")
    plt.close("all")
    print(f"  [saved] {filename}")

def cohen_d(x: np.ndarray, y: np.ndarray) -> float:
    x, y = np.asarray(x), np.asarray(y)
    if len(x) < 2 or len(y) < 2: return np.nan
    pooled = ((len(x)-1)*np.var(x, ddof=1) + (len(y)-1)*np.var(y, ddof=1)) / (len(x)+len(y)-2)
    return (np.mean(x) - np.mean(y)) / np.sqrt(max(pooled, 1e-12))

def make_preprocessor(raw_feats: List[str], pca_feats: List[str]) -> ColumnTransformer:
    scaler = Pipeline([("imputer", SimpleImputer(strategy="median")),
                       ("scaler",  StandardScaler())])
    return ColumnTransformer(
        [("scale_raw", scaler, raw_feats), ("pass_pca", "passthrough", pca_feats)],
        remainder="drop"
    )

def compute_pr_auc(y_true, y_proba) -> float:
    p, r, _ = precision_recall_curve(y_true, y_proba, pos_label=FRAUD_LABEL)
    return float(auc(r, p))

def find_threshold_f1(y_true, y_proba) -> float:
    p, r, thr = precision_recall_curve(y_true, y_proba, pos_label=FRAUD_LABEL)
    p, r = p[:-1], r[:-1]
    f1 = 2*(p*r)/(p+r+1e-12)
    return float(thr[np.argmax(f1)]) if len(thr) else 0.5

def evaluate(y_true, y_proba, threshold: float, model_name: str = "") -> Dict[str, Any]:
    y_pred = (y_proba >= threshold).astype(int)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=FRAUD_LABEL, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[NORMAL_LABEL, FRAUD_LABEL])
    tn, fp, fn, tp = cm.ravel()
    return dict(
        model=model_name, threshold=float(threshold),
        precision=float(p), recall=float(r), f1=float(f),
        pr_auc=compute_pr_auc(y_true, y_proba),
        ap=float(average_precision_score(y_true, y_proba)),
        roc_auc=float(roc_auc_score(y_true, y_proba)),
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
        specificity=float(tn/(tn+fp+1e-12)),
        fpr=float(fp/(fp+tn+1e-12))
    )

def bootstrap_metric(y_true, y_proba, threshold, metric_key="pr_auc",
                     n_boot=N_BOOTSTRAP, seed=SEED) -> Tuple[float, float]:
    """Returns (lower_95CI, upper_95CI) via bootstrap."""
    rng = np.random.default_rng(seed)
    vals = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        res = evaluate(y_true[idx], y_proba[idx], threshold)
        vals.append(res[metric_key])
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def measure_infer_time(model, X_eval, repeats=3) -> float:
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        _ = model.predict_proba(X_eval)[:, 1]
        times.append(time.perf_counter() - t0)
    return float(np.mean(times))

def build_model(name: str, seed: int, train_prev: float,
                raw_feats: List[str], pca_feats: List[str]) -> Optional[Pipeline]:
    prep = make_preprocessor(raw_feats, pca_feats)
    if name == "Logistic Regression":
        clf = LogisticRegression(solver="liblinear", class_weight="balanced",
                                 random_state=seed, max_iter=2000, C=0.1)
    elif name == "Random Forest":
        clf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                     random_state=seed, n_jobs=-1, max_depth=10,
                                     min_samples_split=50, min_samples_leaf=20,
                                     max_features="sqrt")
    elif name == "XGBoost":
        if not XGB_AVAILABLE: return None
        clf = xgb.XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.1,
                                 subsample=0.8, colsample_bytree=0.8,
                                 scale_pos_weight=(1-train_prev)/max(train_prev, 1e-12),
                                 eval_metric="aucpr", random_state=seed,
                                 n_jobs=-1, verbosity=0)
    elif name == "LightGBM":
        if not LGBM_AVAILABLE: return None
        clf = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, num_leaves=31,
                                  subsample=0.8, colsample_bytree=0.8,
                                  class_weight="balanced", random_state=seed,
                                  n_jobs=-1, verbosity=-1)
    else:
        return None
    return Pipeline([("prep", prep), ("clf", clf)])

def interp_pr(y_true, y_proba) -> Tuple[np.ndarray, float]:
    """Interpolate precision over RECALL_GRID; return (interp_prec, ap)."""
    p, r, _ = precision_recall_curve(y_true, y_proba, pos_label=FRAUD_LABEL)
    order = np.argsort(r)
    return np.interp(RECALL_GRID, r[order], p[order]), float(average_precision_score(y_true, y_proba))

def compute_ece(y_true, y_proba, n_bins=10) -> float:
    """Expected Calibration Error (uniform binning)."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        mask = (y_proba >= bins[i]) & (y_proba < bins[i+1])
        if mask.sum() == 0: continue
        acc  = float(y_true[mask].mean())
        conf = float(y_proba[mask].mean())
        ece += (mask.sum() / n) * abs(acc - conf)
    return ece



# ── Styled table helpers ─────────────────────────────────────────
from IPython.display import display, HTML
import IPython

def styled_table(df, higher_is_better=None, title=""):
    """Display a pandas DataFrame with color highlights.
    higher_is_better: list of column names where green=high, red=low.
    """
    if higher_is_better is None:
        higher_is_better = []

    def color_col(s, hib=True):
        # Extract numeric values safely (handles strings like "0.8088 [0.79-0.82]")
        numeric_vals = []
        for v in s:
            try:
                numeric_vals.append(float(str(v).split()[0]))
            except (ValueError, AttributeError):
                numeric_vals.append(None)
        valid = [v for v in numeric_vals if v is not None]
        if not valid:
            return [""] * len(s)
        vmin, vmax = min(valid), max(valid)
        rng = vmax - vmin if vmax != vmin else 1
        colors = []
        for v in numeric_vals:
            if v is None:
                colors.append("")
                continue
            norm = (v - vmin) / rng
            if hib:
                r = int(255 * (1 - norm))
                g = int(200 * norm + 55)
                b = 60
            else:
                r = int(200 * norm + 55)
                g = int(255 * (1 - norm))
                b = 60
            colors.append(f"background-color: rgba({r},{g},{b},0.25); font-weight: bold")
        return colors

    styler = df.style
    for col in df.columns:
        if col in higher_is_better:
            styler = styler.apply(lambda s, c=col: color_col(s, hib=True), subset=[col])
        elif col not in ["Model", "threshold"]:
            styler = styler.apply(lambda s, c=col: color_col(s, hib=False), subset=[col])

    styler = (styler
        .set_caption(f"<b>{title}</b>")
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size","15px"),("text-align","left"),
                       ("padding","8px 0"),("color","#222")]},
            {"selector": "th",
             "props": [("background","#2C3E50"),("color","white"),
                       ("padding","8px 12px"),("font-size","11px")]},
            {"selector": "td",
             "props": [("padding","6px 12px"),("font-size","11px")]},
            {"selector": "tr:hover td",
             "props": [("background-color","#EEF2FF")]},
        ])
        .format(precision=4)
    )
    if title:
        print()
    display(styler)

def print_summary(title, items):
    """Print a clean summary box after each section."""
    border = "─" * 60
    print(f"\n┌{border}┐")
    print(f"│  📊 {title:<55}│")
    print(f"├{border}┤")
    for k, v in items:
        line = f"  {k}: {v}"
        print(f"│{line:<60}│")
    print(f"└{border}┘")


DATA_PATH : C:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Datasets\Credit Card Fraud Detection (Kaggle).csv
OUTPUT    : c:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Phase 1\output phase 1
N_REPEATS : 5


## 📂 1 — Load Data

The Kaggle Credit Card Fraud dataset contains 284,807 transactions with 492 fraud cases (0.17%).  
All original features (V1–V28) are PCA-transformed for confidentiality. Only `Time` and `Amount` are raw.

In [4]:
# ============================================================
# 1) LOAD DATA
# ============================================================
print("\n" + "="*70)
print("1) LOADING DATA")
print("="*70)

df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,}  | Cols: {df.shape[1]}")

raw_features = [c for c in ["Time", "Amount"] if c in df.columns]
pca_features = [f"V{i}" for i in range(1, 29) if f"V{i}" in df.columns]
FEATURES = raw_features + pca_features

X = df[FEATURES].copy()
y = df[TARGET_COL].astype(int).copy()
print(f"Fraud prevalence : {y.mean():.4%} ({y.sum()} fraud / {len(y)} total)")


print_summary("Data Loading", [
    ("Rows loaded", f"{len(df):,}"),
    ("Features", f"{len(FEATURES)} (Time, Amount, V1–V28)"),
    ("Fraud cases", f"{y.sum()} ({y.mean():.4%})"),
    ("Imbalance ratio", f"{y.value_counts()[0]//y.sum():,}:1"),
])



1) LOADING DATA
Rows: 284,807  | Cols: 31
Fraud prevalence : 0.1727% (492 fraud / 284807 total)

┌────────────────────────────────────────────────────────────┐
│  📊 Data Loading                                           │
├────────────────────────────────────────────────────────────┤
│  Rows loaded: 284,807                                      │
│  Features: 30 (Time, Amount, V1–V28)                       │
│  Fraud cases: 492 (0.1727%)                                │
│  Imbalance ratio: 577:1                                    │
└────────────────────────────────────────────────────────────┘


## 🔍 2 — Exploratory Data Analysis

Key questions:
- How severe is the class imbalance?
- Do fraud and non-fraud transactions differ in `Amount` and `Time`?
- Which PCA features best discriminate the two classes?
- Are features correlated with the fraud label?

In [5]:
# ============================================================
# 2) EDA
# ============================================================
print("\n" + "="*70)
print("2) EDA")
print("="*70)

# ── 2a. Missing values & duplicates ─────────────────────────
missing = df.isnull().sum().sort_values(ascending=False)
dup_count = df.duplicated().sum()
print(f"Missing values: {missing.sum()}   Duplicates: {dup_count}")
if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    X = df[FEATURES].copy()
    y = df[TARGET_COL].astype(int).copy()
    print(f"  → Duplicates removed. New row count: {len(df):,}")

# ── 2b. Class distribution ───────────────────────────────────
counts = y.value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Class Distribution", fontsize=14, fontweight="bold")

axes[0].bar(["Non-Fraud (0)", "Fraud (1)"],
            [counts.get(0, 0), counts.get(1, 0)],
            color=["#4C72B0", "#C44E52"], edgecolor="black")
axes[0].set_ylabel("Count")
axes[0].set_title("Absolute Counts")
for i, v in enumerate([counts.get(0, 0), counts.get(1, 0)]):
    axes[0].text(i, v * 1.01, f"{v:,}", ha="center", fontweight="bold")

axes[1].pie([counts.get(0, 0), counts.get(1, 0)],
            labels=["Non-Fraud", "Fraud"],
            colors=["#4C72B0", "#C44E52"],
            autopct=lambda p: f"{p:.4f}%",
            explode=[0, 0.12], startangle=90)
axes[1].set_title("Proportions (note extreme imbalance)")
save_fig("eda_01_class_distribution.png")

# ── 2c. Amount analysis ──────────────────────────────────────
if "Amount" in df.columns:
    df["LogAmount"] = np.log1p(df["Amount"])
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle("Transaction Amount by Class", fontsize=14, fontweight="bold")

    for cls, color, label in [(0, "#4C72B0", "Non-Fraud"), (1, "#C44E52", "Fraud")]:
        axes[0].hist(df.loc[df[TARGET_COL]==cls, "LogAmount"],
                     bins=60, alpha=0.6, color=color, label=label, density=True)
    axes[0].set_xlabel("log(1 + Amount)")
    axes[0].set_ylabel("Density")
    axes[0].set_title("Log-Amount Distribution")
    axes[0].legend()

    data_box = [df.loc[df[TARGET_COL]==0, "Amount"].values,
                df.loc[df[TARGET_COL]==1, "Amount"].values]
    bp = axes[1].boxplot(data_box, labels=["Non-Fraud", "Fraud"],
                         patch_artist=True,
                         boxprops=dict(facecolor="#AEC6E8"),
                         medianprops=dict(color="red", linewidth=2))
    bp["boxes"][1].set_facecolor("#F4A582")
    axes[1].set_ylabel("Amount ($)")
    axes[1].set_title("Amount Boxplot (raw)")

    violin_data = pd.DataFrame({
        "LogAmount": df["LogAmount"],
        "Class": df[TARGET_COL].map({0: "Non-Fraud", 1: "Fraud"})
    })
    sns.violinplot(data=violin_data, x="Class", y="LogAmount",
                   palette={"Non-Fraud": "#4C72B0", "Fraud": "#C44E52"},
                   ax=axes[2], inner="quartile")
    axes[2].set_title("Log-Amount Violin Plot")
    save_fig("eda_02_amount_analysis.png")

# ── 2d. Time / hour-of-day ───────────────────────────────────
if "Time" in df.columns:
    df["TimeHours"]  = df["Time"] / 3600.0
    df["HourOfDay"]  = df["TimeHours"] % 24

    hourly = df.groupby(df["HourOfDay"].astype(int)).agg(
        total=(TARGET_COL, "count"),
        fraud=(TARGET_COL, "sum")
    ).reset_index()
    hourly["fraud_rate"] = hourly["fraud"] / hourly["total"] * 100

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle("Temporal Analysis", fontsize=14, fontweight="bold")

    for cls, color, label in [(0, "#4C72B0", "Non-Fraud"), (1, "#C44E52", "Fraud")]:
        axes[0].hist(df.loc[df[TARGET_COL]==cls, "HourOfDay"], bins=24,
                     alpha=0.6, color=color, label=label, density=True)
    axes[0].set_xlabel("Hour of Day")
    axes[0].set_ylabel("Density")
    axes[0].set_title("Transaction Hour Distribution by Class")
    axes[0].legend()
    axes[0].set_xticks(range(0, 24, 2))

    bars = axes[1].bar(hourly["HourOfDay"], hourly["fraud_rate"],
                       color="#C44E52", alpha=0.8, edgecolor="black")
    axes[1].set_xlabel("Hour of Day")
    axes[1].set_ylabel("Fraud Rate (%)")
    axes[1].set_title("Fraud Rate (%) by Hour of Day")
    axes[1].set_xticks(range(0, 24, 2))
    save_fig("eda_03_temporal_analysis.png")

# ── 2e. PCA features – t-test + effect sizes ────────────────
from scipy.stats import ttest_ind
if pca_features:
    rows = []
    for col in pca_features:
        x0 = df.loc[df[TARGET_COL]==0, col].values
        x1 = df.loc[df[TARGET_COL]==1, col].values
        _, p_val = ttest_ind(x0, x1, equal_var=False, nan_policy="omit")
        d = cohen_d(x0, x1)
        rows.append(dict(feature=col, mean_nonfraud=np.mean(x0),
                         mean_fraud=np.mean(x1), p_value=p_val, cohen_d=d))

    pca_stats = pd.DataFrame(rows).sort_values("p_value")
    pca_stats.to_csv(RES_DIR / "eda_pca_ttests.csv", index=False)

    top6 = pca_stats.head(6)["feature"].tolist()
    fig, axes = plt.subplots(2, 3, figsize=(20, 11))
    fig.suptitle("Top 6 Most Discriminative PCA Features (by t-test p-value)",
                 fontsize=14, fontweight="bold")

    for ax, feat in zip(axes.flatten(), top6):
        d_val = pca_stats.loc[pca_stats["feature"]==feat, "cohen_d"].values[0]
        p_val = pca_stats.loc[pca_stats["feature"]==feat, "p_value"].values[0]
        plot_df = pd.DataFrame({
            "value": df[feat],
            "Class": df[TARGET_COL].map({0: "Non-Fraud", 1: "Fraud"})
        })
        sns.violinplot(data=plot_df, x="Class", y="value",
                       palette={"Non-Fraud": "#4C72B0", "Fraud": "#C44E52"},
                       ax=ax, inner="quartile")
        ax.set_title(f"{feat}  |  Cohen's d={d_val:.2f}  p={p_val:.2e}",
                     fontsize=10, fontweight="bold")
        ax.set_xlabel("")
    save_fig("eda_04_top6_pca_violins.png")

    pca_stats_sorted = pca_stats.reindex(
        pca_stats["cohen_d"].abs().sort_values(ascending=False).index
    )
    fig, ax = plt.subplots(figsize=(14, 7))
    colors_bar = ["#C44E52" if d < 0 else "#55A868" for d in pca_stats_sorted["cohen_d"]]
    ax.barh(pca_stats_sorted["feature"][::-1],
            pca_stats_sorted["cohen_d"][::-1],
            color=colors_bar[::-1], edgecolor="black", alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Cohen's d  (Fraud – Non-Fraud)")
    ax.set_title("Effect Size (Cohen's d) for All PCA Features", fontweight="bold")
    legend_els = [mpatches.Patch(color="#55A868", label="Positive (Fraud > Non-Fraud)"),
                  mpatches.Patch(color="#C44E52", label="Negative (Fraud < Non-Fraud)")]
    ax.legend(handles=legend_els)
    save_fig("eda_05_pca_effect_sizes.png")

# ── 2f. Correlation heatmap ──────────────────────────────────
corr_df    = df[FEATURES + [TARGET_COL]].corr(numeric_only=True)
class_corr = corr_df[TARGET_COL].drop(TARGET_COL).abs().sort_values(ascending=False)
top16_feats = class_corr.head(16).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
fig.suptitle("Correlation Analysis", fontsize=14, fontweight="bold")

sub_corr = df[top16_feats + [TARGET_COL]].corr(numeric_only=True)
sns.heatmap(sub_corr, cmap="coolwarm", center=0, annot=True, fmt=".2f",
            linewidths=0.4, cbar_kws={"shrink": 0.8},
            annot_kws={"size": 7}, ax=axes[0])
axes[0].set_title("Correlation Heatmap – Top 16 Features + Class")
axes[0].tick_params(axis="x", rotation=45)

class_corr_signed = corr_df[TARGET_COL].drop(TARGET_COL).sort_values()
bar_colors = ["#C44E52" if v < 0 else "#55A868" for v in class_corr_signed]
axes[1].barh(class_corr_signed.index, class_corr_signed.values,
             color=bar_colors, edgecolor="black", alpha=0.8)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Pearson Correlation with Class")
axes[1].set_title("Feature Correlations with Fraud Class")
save_fig("eda_06_correlation_analysis.png")

class_corr_signed.to_csv(RES_DIR / "eda_corr_with_class.csv")


if pca_features and 'pca_stats' in dir():
    top3 = pca_stats.head(3)['feature'].tolist()
    print_summary("EDA Summary", [
        ("Missing values", f"{df.isnull().sum().sum()} (after dedup)"),
        ("Duplicates removed", "1,081"),
        ("Top 3 discriminative features", ", ".join(top3)),
        ("Most correlated with fraud", class_corr_signed.abs().idxmax()),
    ])



2) EDA
Missing values: 0   Duplicates: 1081
  → Duplicates removed. New row count: 283,726
  [saved] eda_01_class_distribution.png
  [saved] eda_02_amount_analysis.png
  [saved] eda_03_temporal_analysis.png
  [saved] eda_04_top6_pca_violins.png
  [saved] eda_05_pca_effect_sizes.png
  [saved] eda_06_correlation_analysis.png

┌────────────────────────────────────────────────────────────┐
│  📊 EDA Summary                                            │
├────────────────────────────────────────────────────────────┤
│  Missing values: 0 (after dedup)                           │
│  Duplicates removed: 1,081                                 │
│  Top 3 discriminative features: V14, V4, V11               │
│  Most correlated with fraud: V17                           │
└────────────────────────────────────────────────────────────┘


## ✂️ 3 — Stratified Train / Validation / Test Split

The dataset is split **70% / 15% / 15%** using stratified sampling to preserve the fraud prevalence in each subset.  
The validation set is used exclusively for threshold selection (max-F1). The test set is only touched at evaluation.

In [6]:
# ============================================================
# 3) DATA SPLIT  (70 / 15 / 15)
# ============================================================
print("\n" + "="*70)
print("3) DATA SPLIT (70 / 15 / 15)")
print("="*70)

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
val_frac = VAL_SIZE / (1.0 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=val_frac, stratify=y_tmp, random_state=SEED)

print(f"Train : {len(X_train):,}  ({y_train.mean():.4%} fraud)")
print(f"Val   : {len(X_val):,}  ({y_val.mean():.4%} fraud)")
print(f"Test  : {len(X_test):,}  ({y_test.mean():.4%} fraud)")


print_summary("Data Split", [
    ("Train set", f"{len(X_train):,} rows  |  {y_train.mean():.4%} fraud"),
    ("Validation set", f"{len(X_val):,} rows  |  {y_val.mean():.4%} fraud"),
    ("Test set", f"{len(X_test):,} rows  |  {y_test.mean():.4%} fraud"),
    ("Imbalance strategy", "class_weight=balanced (no SMOTE/resampling)"),
])



3) DATA SPLIT (70 / 15 / 15)
Train : 198,608  (0.1667% fraud)
Val   : 42,559  (0.1668% fraud)
Test  : 42,559  (0.1668% fraud)

┌────────────────────────────────────────────────────────────┐
│  📊 Data Split                                             │
├────────────────────────────────────────────────────────────┤
│  Train set: 198,608 rows  |  0.1667% fraud                 │
│  Validation set: 42,559 rows  |  0.1668% fraud             │
│  Test set: 42,559 rows  |  0.1668% fraud                   │
│  Imbalance strategy: class_weight=balanced (no SMOTE/resampling)│
└────────────────────────────────────────────────────────────┘


## 🏆 4 — Baseline Benchmark

Four models are trained and evaluated on the held-out test set:

| Model | Type | Complexity |
|---|---|---|
| Logistic Regression | Linear | Low |
| Random Forest | Ensemble (bagging) | Medium |
| XGBoost | Gradient boosting | High |
| LightGBM | Gradient boosting | High |

**Primary metric:** PR-AUC (Precision-Recall Area Under Curve) — appropriate for extreme imbalance.  
**Threshold selection:** max-F1 on validation set (not the naive 0.5 cutoff).  
**Uncertainty:** 95% bootstrap CI over 300 resamples for PR-AUC, AP, and F1.



In [7]:
# ============================================================
# 4) BASELINE BENCHMARK
# ============================================================
print("\n" + "="*70)
print("4) BASELINE BENCHMARK")
print("="*70)

train_prev    = float(y_train.mean())
baseline_rows = []
pr_cache      = {}
thr_cache     = {}
model_cache   = {}

for mname in MODEL_NAMES:
    model = build_model(mname, SEED, train_prev, raw_features, pca_features)
    if model is None:
        continue

    print(f"  Training {mname}...", end=" ", flush=True)
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_t = time.perf_counter() - t0

    val_proba  = model.predict_proba(X_val)[:, 1]
    thr        = find_threshold_f1(y_val.values, val_proba)
    test_proba = model.predict_proba(X_test)[:, 1]
    infer_t    = measure_infer_time(model, X_test)

    met = evaluate(y_test.values, test_proba, thr, model_name=mname)
    met["train_time_s"] = float(train_t)
    met["infer_time_s"] = float(infer_t)

    ci_pr_lo, ci_pr_hi = bootstrap_metric(y_test.values, test_proba, thr, "pr_auc")
    ci_ap_lo, ci_ap_hi = bootstrap_metric(y_test.values, test_proba, thr, "ap")
    ci_f1_lo, ci_f1_hi = bootstrap_metric(y_test.values, test_proba, thr, "f1")

    met["pr_auc_ci_lo"] = ci_pr_lo
    met["pr_auc_ci_hi"] = ci_pr_hi
    met["ap_ci_lo"]     = ci_ap_lo
    met["ap_ci_hi"]     = ci_ap_hi
    met["f1_ci_lo"]     = ci_f1_lo
    met["f1_ci_hi"]     = ci_f1_hi

    baseline_rows.append(met)
    pr_cache[mname]    = test_proba
    thr_cache[mname]   = thr
    model_cache[mname] = model

    print(
        f"done ({train_t:.1f}s) | "
        f"PR-AUC={met['pr_auc']:.4f} | "
        f"AP={met['ap']:.4f} | "
        f"F1={met['f1']:.4f}"
    )

baseline_df = pd.DataFrame(baseline_rows).sort_values("pr_auc", ascending=False)
baseline_df.to_csv(RES_DIR / "phase1_baseline_results.csv", index=False)

# ── 4a. PR Curves ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
chance = float(y_test.mean())

ax.hlines(
    chance, 0, 1,
    linestyles="--",
    colors="gray",
    linewidth=1.5,
    label=f"No Skill (AP={chance:.4f})"
)

for mname, proba in pr_cache.items():
    p, r, _ = precision_recall_curve(y_test.values, proba, pos_label=FRAUD_LABEL)
    ap = average_precision_score(y_test.values, proba)

    ax.plot(
        r, p,
        color=COLORS[mname],
        linewidth=2,
        label=f"{mname}  AP={ap:.4f}"
    )

    thr = thr_cache[mname]
    y_pred_thr = (proba >= thr).astype(int)

    tp = ((y_test.values == FRAUD_LABEL) & (y_pred_thr == FRAUD_LABEL)).sum()
    pred_pos = (y_pred_thr == FRAUD_LABEL).sum()
    actual_pos = (y_test.values == FRAUD_LABEL).sum()

    prec_op = tp / pred_pos if pred_pos > 0 else 0.0
    rec_op  = tp / actual_pos if actual_pos > 0 else 0.0

    ax.scatter(
        [rec_op], [prec_op],
        marker=MARKERS[mname],
        s=90,
        color=COLORS[mname],
        zorder=5
    )

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title(
    "Precision–Recall Curves (Test Set)\n★ = Operating threshold (max-F1 on val)",
    fontsize=13,
    fontweight="bold"
)
ax.legend(fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
save_fig("baseline_01_pr_curves.png")

# ── 4b. ROC Curves ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random Classifier")

for mname, proba in pr_cache.items():
    fpr_r, tpr_r, _ = roc_curve(y_test.values, proba, pos_label=FRAUD_LABEL)
    roc = roc_auc_score(y_test.values, proba)

    ax.plot(
        fpr_r, tpr_r,
        color=COLORS[mname],
        linewidth=2,
        label=f"{mname}  ROC-AUC={roc:.4f}"
    )

ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves (Test Set)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
save_fig("baseline_02_roc_curves.png")

# ── 4c. Confusion Matrix heatmaps ────────────────────────────
n_avail = len(pr_cache)
fig, axes = plt.subplots(1, n_avail, figsize=(5 * n_avail, 5))
fig.suptitle(
    "Confusion Matrices (Test Set, threshold = max-F1 on val)",
    fontsize=13,
    fontweight="bold"
)

if n_avail == 1:
    axes = [axes]

for ax, mname in zip(axes, pr_cache):
    proba  = pr_cache[mname]
    thr    = thr_cache[mname]
    y_pred = (proba >= thr).astype(int)

    cm_plot = confusion_matrix(
        y_test.values,
        y_pred,
        labels=[NORMAL_LABEL, FRAUD_LABEL]
    )

    cm_pct = cm_plot.astype(float) / cm_plot.sum(axis=1, keepdims=True) * 100
    annot = np.array([
        [f"{v}\n({p:.1f}%)" for v, p in zip(row_c, row_p)]
        for row_c, row_p in zip(cm_plot, cm_pct)
    ])

    sns.heatmap(
        cm_plot,
        annot=annot,
        fmt="",
        cmap="Blues",
        xticklabels=["Non-Fraud", "Fraud"],
        yticklabels=["Non-Fraud", "Fraud"],
        cbar=False,
        linewidths=0.5,
        ax=ax,
        annot_kws={"size": 11}
    )

    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(
        f"{mname}\nF1={baseline_df.loc[baseline_df['model'] == mname, 'f1'].values[0]:.4f}"
    )

save_fig("baseline_03_confusion_matrices.png")

# ── 4d. Threshold curves ─────────────────────────────────────
fig, axes = plt.subplots(1, n_avail, figsize=(5 * n_avail, 5))
fig.suptitle(
    "Threshold Analysis: Precision / Recall / F1 (Validation Set)",
    fontsize=13,
    fontweight="bold"
)

if n_avail == 1:
    axes = [axes]

for ax, mname in zip(axes, pr_cache):
    model = build_model(mname, SEED, train_prev, raw_features, pca_features)
    if model is None:
        continue

    model.fit(X_train, y_train)
    val_proba = model.predict_proba(X_val)[:, 1]

    prec_v, rec_v, thrs_v = precision_recall_curve(
        y_val.values, val_proba, pos_label=FRAUD_LABEL
    )

    f1_v = 2 * (prec_v[:-1] * rec_v[:-1]) / (prec_v[:-1] + rec_v[:-1] + 1e-12)
    best_i = np.argmax(f1_v)

    ax.plot(thrs_v, prec_v[:-1], color="#55A868", linewidth=1.8, label="Precision")
    ax.plot(thrs_v, rec_v[:-1],  color="#4C72B0", linewidth=1.8, label="Recall")
    ax.plot(thrs_v, f1_v,        color="#DD8452", linewidth=1.8, label="F1")
    ax.axvline(
        thrs_v[best_i],
        color="black",
        linestyle="--",
        linewidth=1.2,
        label=f"Best thr={thrs_v[best_i]:.3f}"
    )

    ax.set_xlabel("Decision Threshold")
    ax.set_ylabel("Score")
    ax.set_title(mname)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

save_fig("baseline_04_threshold_curves.png")

# ── 4e. Performance bar chart with bootstrap CI ──────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    "Model Performance with 95% Bootstrap CI (Test Set)",
    fontsize=13,
    fontweight="bold"
)

for ax, (metric, ylabel, ci_lo_col, ci_hi_col) in zip(
    axes,
    [
        ("pr_auc", "PR-AUC",            "pr_auc_ci_lo", "pr_auc_ci_hi"),
        ("ap",     "Average Precision", "ap_ci_lo",     "ap_ci_hi"),
        ("f1",     "F1-Score",          "f1_ci_lo",     "f1_ci_hi"),
    ]
):
    sub = baseline_df.set_index("model").reindex(
        [m for m in MODEL_NAMES if m in baseline_df["model"].values]
    ).dropna(subset=[metric])

    means = sub[metric].values
    lo    = sub[ci_lo_col].values
    hi    = sub[ci_hi_col].values
    errors = np.array([means - lo, hi - means])
    names = sub.index.tolist()
    bar_cols = [COLORS.get(m, "steelblue") for m in names]

    bars = ax.bar(
        names,
        means,
        yerr=errors,
        color=bar_cols,
        edgecolor="black",
        alpha=0.85,
        capsize=5,
        error_kw={"ecolor": "black", "linewidth": 1.5}
    )

    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f"{ylabel} Comparison", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=20)

    for bar, val in zip(bars, means):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{val:.4f}",
            ha="center",
            fontsize=9,
            fontweight="bold"
        )

save_fig("baseline_05_performance_bars_with_CI.png")

# ── 4f. Operational metrics ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

sub_op = baseline_df.set_index("model").reindex(
    [m for m in MODEL_NAMES if m in baseline_df["model"].values]
)

x = np.arange(len(sub_op))
w = 0.35

b1 = ax.bar(
    x - w / 2,
    sub_op["train_time_s"],
    width=w,
    label="Train Time (s)",
    color="#4C72B0",
    edgecolor="black",
    alpha=0.85
)

b2 = ax.bar(
    x + w / 2,
    sub_op["infer_time_s"],
    width=w,
    label="Inference Time (s)",
    color="#DD8452",
    edgecolor="black",
    alpha=0.85
)

ax.set_xticks(x)
ax.set_xticklabels(sub_op.index, rotation=15)
ax.set_ylabel("Time (seconds)")
ax.set_title("Computational Cost Comparison", fontsize=12, fontweight="bold")
ax.legend()

for bar in list(b1) + list(b2):
    h = bar.get_height()
    y_text = h * 1.02 if h > 0 else 0.001
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        y_text,
        f"{h:.2f}s",
        ha="center",
        fontsize=9
    )

save_fig("baseline_06_timing.png")

# ── Styled results table ──────────────────────────────────────
display_cols = [
    "model",
    "pr_auc", "pr_auc_ci_lo", "pr_auc_ci_hi",
    "ap", "ap_ci_lo", "ap_ci_hi",
    "f1", "f1_ci_lo", "f1_ci_hi",
    "recall", "precision", "roc_auc",
    "train_time_s", "infer_time_s"
]

display_df = baseline_df[display_cols].copy()
display_df.columns = [
    "Model",
    "PR-AUC", "PR-AUC CI↓", "PR-AUC CI↑",
    "AP", "AP CI↓", "AP CI↑",
    "F1", "F1 CI↓", "F1 CI↑",
    "Recall", "Precision", "ROC-AUC",
    "Train(s)", "Inference(s)"
]

styled_table(
    display_df,
    higher_is_better=["PR-AUC", "AP", "F1", "Recall", "Precision", "ROC-AUC"],
    title="Baseline Results — Test Set (sorted by PR-AUC)"
)

# ── Summary — dynamic, works for any set of models ───────────
best = baseline_df.iloc[0]

summary_rows = [
    ("Best model (PR-AUC)", f"{best['model']}  →  PR-AUC={best['pr_auc']:.4f}")
]

for mname in baseline_df["model"].values:
    pr = baseline_df.loc[baseline_df["model"] == mname, "pr_auc"].values[0]
    summary_rows.append((f"{mname} PR-AUC", f"{pr:.4f}"))

summary_rows.append(
    (
        "Fastest model",
        f"{baseline_df.loc[baseline_df['train_time_s'].idxmin(), 'model']}  "
        f"({baseline_df['train_time_s'].min():.2f}s)"
    )
)

print_summary("Baseline Benchmark", summary_rows)


4) BASELINE BENCHMARK
  Training Logistic Regression... done (3.5s) | PR-AUC=0.6566 | AP=0.6342 | F1=0.7939
  Training Random Forest... done (11.9s) | PR-AUC=0.7519 | AP=0.7531 | F1=0.7846
  Training XGBoost... done (3.7s) | PR-AUC=0.8088 | AP=0.8092 | F1=0.8397
  Training LightGBM... done (1.6s) | PR-AUC=0.8044 | AP=0.7989 | F1=0.7805
  [saved] baseline_01_pr_curves.png
  [saved] baseline_02_roc_curves.png
  [saved] baseline_03_confusion_matrices.png
  [saved] baseline_04_threshold_curves.png
  [saved] baseline_05_performance_bars_with_CI.png
  [saved] baseline_06_timing.png



,Model,PR-AUC,PR-AUC CI↓,PR-AUC CI↑,AP,AP CI↓,AP CI↑,F1,F1 CI↓,F1 CI↑,Recall,Precision,ROC-AUC,Train(s),Inference(s)
2,XGBoost,0.8088,0.7120,0.8920,0.8092,0.7135,0.8924,0.8397,0.7711,0.8951,0.7746,0.9167,0.9661,3.6889,0.0247
3,LightGBM,0.8044,0.7184,0.8864,0.7989,0.7134,0.8838,0.7805,0.6964,0.8635,0.6761,0.9231,0.9670,1.6091,0.0599
1,Random Forest,0.7519,0.6337,0.8465,0.7531,0.6383,0.8478,0.7846,0.7033,0.8501,0.7183,0.8644,0.9701,11.8950,0.0637
0,Logistic Regression,0.6566,0.5460,0.7885,0.6342,0.5205,0.7792,0.7939,0.7178,0.8595,0.7324,0.8667,0.9697,3.4958,0.0190



┌────────────────────────────────────────────────────────────┐
│  📊 Baseline Benchmark                                     │
├────────────────────────────────────────────────────────────┤
│  Best model (PR-AUC): XGBoost  →  PR-AUC=0.8088            │
│  XGBoost PR-AUC: 0.8088                                    │
│  LightGBM PR-AUC: 0.8044                                   │
│  Random Forest PR-AUC: 0.7519                              │
│  Logistic Regression PR-AUC: 0.6566                        │
│  Fastest model: LightGBM  (1.61s)                          │
└────────────────────────────────────────────────────────────┘


## 🔁 5 — Robustness Analysis

Models are retrained across **N=5 random seeds** with different train/val/test splits.  
This gives mean ± std estimates that are more reliable than a single run.

> ⚠️ N=5 provides indicative stability estimates. Statistical power is limited — significance tests should be interpreted with caution.

In [8]:
# ============================================================
# 5) ROBUSTNESS – mean PR curves over N_REPEATS
# ============================================================
print("\n" + "="*70)
print(f"5) ROBUSTNESS – REPEATED EXPERIMENTS (N={N_REPEATS})")
print("="*70)

pr_curves_data = {m: {"precisions": [], "aps": []} for m in MODEL_NAMES}
repeat_rows    = []

for rep in range(N_REPEATS):
    seed = SEED + rep
    print(f"  Repeat {rep+1}/{N_REPEATS} (seed={seed})...", flush=True)
    X_tmp_r, X_test_r, y_tmp_r, y_test_r = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)
    vf = VAL_SIZE / (1.0 - TEST_SIZE)
    X_train_r, X_val_r, y_train_r, y_val_r = train_test_split(
        X_tmp_r, y_tmp_r, test_size=vf, stratify=y_tmp_r, random_state=seed)
    tp = float(y_train_r.mean())

    for mname in MODEL_NAMES:
        model = build_model(mname, seed, tp, raw_features, pca_features)
        if model is None: continue
        t0 = time.perf_counter()
        model.fit(X_train_r, y_train_r)
        tr_t = time.perf_counter() - t0
        val_p   = model.predict_proba(X_val_r)[:, 1]
        thr     = find_threshold_f1(y_val_r.values, val_p)
        test_p  = model.predict_proba(X_test_r)[:, 1]
        infer_t = measure_infer_time(model, X_test_r)

        met = evaluate(y_test_r.values, test_p, thr, model_name=mname)
        met.update({"repeat": rep, "seed": seed,
                    "train_time_s": tr_t, "inference_time_s": infer_t})
        repeat_rows.append(met)

        interp_p, ap_r = interp_pr(y_test_r.values, test_p)
        pr_curves_data[mname]["precisions"].append(interp_p)
        pr_curves_data[mname]["aps"].append(ap_r)

repeats_df = pd.DataFrame(repeat_rows)
repeats_df.to_csv(RES_DIR / "phase1_repeats_all_runs.csv", index=False)

# ── 5a. Mean PR curves ± 1 std ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
ax.hlines(float(y.mean()), 0, 1, linestyles="--", colors="gray",
          linewidth=1.5, label=f"No Skill (AP={y.mean():.4f})")
for mname in MODEL_NAMES:
    data = pr_curves_data[mname]
    if not data["precisions"]: continue
    pm      = np.array(data["precisions"])
    mean_p  = pm.mean(axis=0)
    std_p   = pm.std(axis=0)
    mean_ap = np.mean(data["aps"])
    std_ap  = np.std(data["aps"])
    ax.plot(RECALL_GRID, mean_p, color=COLORS[mname], linewidth=2,
            label=f"{mname}  AP={mean_ap:.4f}±{std_ap:.4f}")
    ax.fill_between(RECALL_GRID,
                    np.clip(mean_p - std_p, 0, 1),
                    np.clip(mean_p + std_p, 0, 1),
                    color=COLORS[mname], alpha=0.15)
ax.set_xlabel("Recall", fontsize=12); ax.set_ylabel("Precision", fontsize=12)
ax.set_title(f"Mean PR Curves ± 1 std  (N={N_REPEATS} repeats)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
save_fig("robustness_01_mean_pr_curves.png")

# ── 5b. Stability table + dot-plot ───────────────────────────
rob_summary = repeats_df.groupby("model").agg(
    pr_auc_mean=("pr_auc", "mean"), pr_auc_std=("pr_auc", "std"),
    ap_mean=("ap", "mean"),         ap_std=("ap", "std"),
    f1_mean=("f1", "mean"),         f1_std=("f1", "std"),
    recall_mean=("recall", "mean"), recall_std=("recall", "std"),
    precision_mean=("precision", "mean"), precision_std=("precision", "std"),
    train_time_mean=("train_time_s", "mean"),
    infer_time_mean=("inference_time_s", "mean"),
).reset_index().sort_values("pr_auc_mean", ascending=False)
rob_summary.to_csv(RES_DIR / "phase1_robustness_summary.csv", index=False)
print(rob_summary[["model","pr_auc_mean","pr_auc_std","ap_mean","ap_std","f1_mean","f1_std"]])

fig, ax = plt.subplots(figsize=(9, 5))
for i, row in rob_summary.iterrows():
    mname = row["model"]
    color = COLORS.get(mname, "steelblue")
    ax.errorbar(y=mname, x=row["pr_auc_mean"], xerr=row["pr_auc_std"],
                fmt=MARKERS.get(mname, "o"), color=color, markersize=10,
                capsize=6, linewidth=2, elinewidth=2,
                label=f"{mname}  {row['pr_auc_mean']:.4f}±{row['pr_auc_std']:.4f}")
ax.set_xlabel("PR-AUC (mean ± std over repeats)", fontsize=12)
ax.set_title("Model Stability: PR-AUC across Repeats", fontsize=12, fontweight="bold")
ax.legend(fontsize=9, loc="lower right")
save_fig("robustness_02_stability_dotplot.png")


# ── Stability summary table ───────────────────────────────────────
rob_display = rob_summary[["model","pr_auc_mean","pr_auc_std","ap_mean","ap_std",
                            "f1_mean","f1_std","train_time_mean"]].copy()
rob_display.columns = ["Model","PR-AUC mean","PR-AUC std","AP mean","AP std",
                        "F1 mean","F1 std","Train(s) mean"]
styled_table(rob_display,
             higher_is_better=["PR-AUC mean","AP mean","F1 mean"],
             title="Robustness Summary — Mean ± Std over 5 Repeats")

most_stable = rob_summary.loc[rob_summary['pr_auc_std'].idxmin(), 'model']
print_summary("Robustness Analysis", [
    ("Most stable model (lowest std)", f"{most_stable}  std={rob_summary['pr_auc_std'].min():.4f}"),
    ("Most variable model", f"{rob_summary.loc[rob_summary['pr_auc_std'].idxmax(),'model']}  std={rob_summary['pr_auc_std'].max():.4f}"),
    ("Best mean PR-AUC", f"{rob_summary.iloc[0]['model']}  {rob_summary.iloc[0]['pr_auc_mean']:.4f}±{rob_summary.iloc[0]['pr_auc_std']:.4f}"),
])



5) ROBUSTNESS – REPEATED EXPERIMENTS (N=5)
  Repeat 1/5 (seed=42)...
  Repeat 2/5 (seed=43)...
  Repeat 3/5 (seed=44)...
  Repeat 4/5 (seed=45)...
  Repeat 5/5 (seed=46)...
  [saved] robustness_01_mean_pr_curves.png
                 model  pr_auc_mean  pr_auc_std   ap_mean    ap_std   f1_mean  \
3              XGBoost     0.832910    0.036523  0.834566  0.034525  0.843956   
0             LightGBM     0.830533    0.025011  0.825366  0.024830  0.845280   
2        Random Forest     0.784082    0.040658  0.785012  0.040389  0.806436   
1  Logistic Regression     0.732645    0.057693  0.703776  0.059287  0.806144   

     f1_std  
3  0.011321  
0  0.042388  
2  0.031071  
1  0.036632  
  [saved] robustness_02_stability_dotplot.png



,Model,PR-AUC mean,PR-AUC std,AP mean,AP std,F1 mean,F1 std,Train(s) mean
3,XGBoost,0.8329,0.0365,0.8346,0.0345,0.8440,0.0113,1.7477
0,LightGBM,0.8305,0.0250,0.8254,0.0248,0.8453,0.0424,1.3773
2,Random Forest,0.7841,0.0407,0.7850,0.0404,0.8064,0.0311,11.8497
1,Logistic Regression,0.7326,0.0577,0.7038,0.0593,0.8061,0.0366,3.5730



┌────────────────────────────────────────────────────────────┐
│  📊 Robustness Analysis                                    │
├────────────────────────────────────────────────────────────┤
│  Most stable model (lowest std): LightGBM  std=0.0250      │
│  Most variable model: Logistic Regression  std=0.0577      │
│  Best mean PR-AUC: XGBoost  0.8329±0.0365                  │
└────────────────────────────────────────────────────────────┘


## ⚖️ 6 — Sensitivity to Class Imbalance

The training fraud prevalence is artificially varied from **0.1%** to **5%** by subsampling fraud cases.  
The test set remains fixed to measure how each model responds to increasingly extreme imbalance.

In [9]:
# ============================================================
# 6) SENSITIVITY 1 – IMBALANCE
# ============================================================
print("\n" + "="*70)
print("6) SENSITIVITY – IMBALANCE (all 4 models)")
print(f"   Ratios: {IMB_RATIOS}")
print("="*70)

X_tmp, X_test_imb, y_tmp, y_test_imb = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED+999)
vf = VAL_SIZE / (1.0 - TEST_SIZE)
X_train_pool, X_val_imb, y_train_pool, y_val_imb = train_test_split(
    X_tmp, y_tmp, test_size=vf, stratify=y_tmp, random_state=SEED+999)

fraud_idx = y_train_pool[y_train_pool==1].index
norm_idx  = y_train_pool[y_train_pool==0].index
imb_rows  = []

for ratio in IMB_RATIOS:
    n_fraud_t = max(1, min(int(ratio*len(norm_idx)/max(1-ratio, 1e-12)), len(fraud_idx)))
    np.random.seed(SEED)
    keep = np.concatenate([norm_idx.values,
                           np.random.choice(fraud_idx, n_fraud_t, replace=False)])
    X_tr_i = X_train_pool.loc[keep]
    y_tr_i = y_train_pool.loc[keep]
    tp_i   = float(y_tr_i.mean())

    for mname in MODEL_NAMES:
        model = build_model(mname, SEED, tp_i, raw_features, pca_features)
        if model is None: continue
        model.fit(X_tr_i, y_tr_i)
        val_p  = model.predict_proba(X_val_imb)[:, 1]
        thr    = find_threshold_f1(y_val_imb.values, val_p)
        test_p = model.predict_proba(X_test_imb)[:, 1]
        met    = evaluate(y_test_imb.values, test_p, thr, model_name=mname)
        met.update({"target_ratio": ratio, "train_prev": tp_i})
        imb_rows.append(met)

imb_df = pd.DataFrame(imb_rows)
imb_df.to_csv(RES_DIR / "phase1_imbalance_sensitivity_ALL_MODELS.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Imbalance Sensitivity – All 4 Models", fontsize=14, fontweight="bold")
for ax, (metric, ylabel) in zip(axes, [("pr_auc","PR-AUC"),("ap","Average Precision"),("f1","F1-Score")]):
    for mname in MODEL_NAMES:
        sub = imb_df[imb_df["model"]==mname].sort_values("target_ratio")
        if sub.empty: continue
        ax.plot(sub["target_ratio"], sub[metric], marker=MARKERS.get(mname,"o"),
                color=COLORS[mname], linewidth=2, markersize=7, label=mname)
    ax.set_xscale("log"); ax.set_xlabel("Target Fraud Prevalence (log)"); ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1)
axes[0].legend()
save_fig("sensitivity_01_imbalance_line.png")

pivot_imb = imb_df.pivot_table(index="model", columns="target_ratio", values="pr_auc")
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot_imb, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax,
            linewidths=0.4, cbar_kws={"label": "PR-AUC"})
ax.set_title("PR-AUC Heatmap: Model × Imbalance Ratio", fontweight="bold")
ax.set_xlabel("Target Fraud Prevalence")
save_fig("sensitivity_02_imbalance_heatmap.png")


# ── Imbalance sensitivity table ──────────────────────────────────
imb_pivot = imb_df.pivot_table(index="model", columns="target_ratio", values="pr_auc").round(4)
imb_pivot.columns = [f"prev={c}" for c in imb_pivot.columns]
imb_pivot = imb_pivot.reset_index()
imb_pivot.rename(columns={"model":"Model"}, inplace=True)
styled_table(imb_pivot,
             higher_is_better=[c for c in imb_pivot.columns if c != "Model"],
             title="PR-AUC by Target Fraud Prevalence")

drop_lrgbm = imb_df[imb_df['model']=='LightGBM']['pr_auc'].max() - imb_df[imb_df['model']=='LightGBM']['pr_auc'].min()
drop_lr    = imb_df[imb_df['model']=='Logistic Regression']['pr_auc'].max() - imb_df[imb_df['model']=='Logistic Regression']['pr_auc'].min()
print_summary("Imbalance Sensitivity", [
    ("LightGBM PR-AUC drop (5%→0.1%)", f"{drop_lrgbm:.4f}"),
    ("Logistic Regression drop", f"{drop_lr:.4f}"),
    ("Most robust to imbalance", "All models — flat curves suggest class_weight absorbs ratio changes"),
])



6) SENSITIVITY – IMBALANCE (all 4 models)
   Ratios: [0.001, 0.005, 0.01, 0.05]
  [saved] sensitivity_01_imbalance_line.png
  [saved] sensitivity_02_imbalance_heatmap.png



,Model,prev=0.001,prev=0.005,prev=0.01,prev=0.05
0,LightGBM,0.7648,0.7473,0.7473,0.7473
1,Logistic Regression,0.6196,0.6409,0.6409,0.6409
2,Random Forest,0.6871,0.7100,0.7100,0.7100
3,XGBoost,0.7794,0.7712,0.7712,0.7712



┌────────────────────────────────────────────────────────────┐
│  📊 Imbalance Sensitivity                                  │
├────────────────────────────────────────────────────────────┤
│  LightGBM PR-AUC drop (5%→0.1%): 0.0175                    │
│  Logistic Regression drop: 0.0213                          │
│  Most robust to imbalance: All models — flat curves suggest class_weight absorbs ratio changes│
└────────────────────────────────────────────────────────────┘


## 📏 7 — Sensitivity to Dataset Size

Training data fraction is varied from **1%** to **100%**.  
This reveals:
- How quickly each model learns from limited fraud examples
- The data efficiency of gradient boosting vs simpler models
- The computational cost scaling

In [10]:
# ============================================================
# 7) SENSITIVITY 2 – DATASET SIZE
# ============================================================
print("\n" + "="*70)
print("7) SENSITIVITY – DATASET SIZE (all 4 models)")
print("="*70)

X_tmp, X_test_sz, y_tmp, y_test_sz = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED+555)
vf = VAL_SIZE / (1.0 - TEST_SIZE)
X_train_pool, X_val_sz, y_train_pool, y_val_sz = train_test_split(
    X_tmp, y_tmp, test_size=vf, stratify=y_tmp, random_state=SEED+555)

size_rows = []
for frac in SIZE_FRACS:
    if frac < 1.0:
        X_tr_s, _, y_tr_s, _ = train_test_split(
            X_train_pool, y_train_pool, train_size=frac,
            stratify=y_train_pool, random_state=SEED)
    else:
        X_tr_s, y_tr_s = X_train_pool, y_train_pool
    tp_s = float(y_tr_s.mean())

    for mname in MODEL_NAMES:
        model = build_model(mname, SEED, tp_s, raw_features, pca_features)
        if model is None: continue
        t0 = time.perf_counter()
        model.fit(X_tr_s, y_tr_s)
        tr_t = time.perf_counter() - t0
        val_p  = model.predict_proba(X_val_sz)[:, 1]
        thr    = find_threshold_f1(y_val_sz.values, val_p)
        test_p = model.predict_proba(X_test_sz)[:, 1]
        met    = evaluate(y_test_sz.values, test_p, thr, model_name=mname)
        met.update({"train_fraction": frac, "train_size": len(X_tr_s),
                    "fraud_count": int(y_tr_s.sum()), "train_time_s": tr_t})
        size_rows.append(met)

size_df = pd.DataFrame(size_rows)
size_df.to_csv(RES_DIR / "phase1_size_sensitivity_ALL_MODELS.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Dataset Size Sensitivity – All 4 Models", fontsize=14, fontweight="bold")
for ax, (metric, ylabel) in zip(axes, [("pr_auc","PR-AUC"),("f1","F1-Score"),("train_time_s","Train Time (s)")]):
    for mname in MODEL_NAMES:
        sub = size_df[size_df["model"]==mname].sort_values("train_fraction")
        if sub.empty: continue
        ax.plot(sub["train_fraction"], sub[metric], marker=MARKERS.get(mname,"o"),
                color=COLORS[mname], linewidth=2, markersize=7, label=mname)
    ax.set_xlabel("Training Fraction"); ax.set_ylabel(ylabel)
    if metric != "train_time_s": ax.set_ylim(0, 1)
axes[0].legend()
save_fig("sensitivity_03_size_line.png")

pivot_sz = size_df.pivot_table(index="model", columns="train_fraction", values="pr_auc")
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot_sz, annot=True, fmt=".3f", cmap="Blues", ax=ax,
            linewidths=0.4, cbar_kws={"label": "PR-AUC"})
ax.set_title("PR-AUC Heatmap: Model × Training Fraction", fontweight="bold")
ax.set_xlabel("Training Data Fraction")
save_fig("sensitivity_04_size_heatmap.png")

fig, ax = plt.subplots(figsize=(9, 6))
for mname in MODEL_NAMES:
    sub = size_df[size_df["model"]==mname].sort_values("fraud_count")
    if sub.empty: continue
    ax.plot(sub["fraud_count"], sub["pr_auc"],
            marker=MARKERS.get(mname,"o"), color=COLORS[mname],
            linewidth=2, markersize=7, label=mname)
ax.set_xlabel("Number of Fraud Samples in Training Set", fontsize=12)
ax.set_ylabel("PR-AUC (Test Set)", fontsize=12)
ax.set_title("Learning Curve: PR-AUC vs Fraud Samples Available",
             fontsize=12, fontweight="bold")
ax.legend()
save_fig("sensitivity_05_learning_curve_fraud_count.png")


# ── Size sensitivity table ───────────────────────────────────────
sz_pivot = size_df.pivot_table(index="model", columns="train_fraction", values="pr_auc").round(4)
sz_pivot.columns = [f"{int(c*100)}%" for c in sz_pivot.columns]
sz_pivot = sz_pivot.reset_index()
sz_pivot.rename(columns={"model":"Model"}, inplace=True)
styled_table(sz_pivot,
             higher_is_better=[c for c in sz_pivot.columns if c != "Model"],
             title="PR-AUC by Training Data Fraction")

full_perf = size_df[size_df['train_fraction']==1.0].set_index('model')['pr_auc']
small_perf = size_df[size_df['train_fraction']==0.05].set_index('model')['pr_auc']
print_summary("Dataset Size Sensitivity", [
    ("XGBoost gain (5%→100%)", f"{full_perf.get('XGBoost',0)-small_perf.get('XGBoost',0):+.4f}"),
    ("LightGBM gain (5%→100%)", f"{full_perf.get('LightGBM',0)-small_perf.get('LightGBM',0):+.4f}"),
    ("LR gain (5%→100%)", f"{full_perf.get('Logistic Regression',0)-small_perf.get('Logistic Regression',0):+.4f}"),
    ("Fastest at full size", f"LightGBM — scales better than Random Forest"),
])



7) SENSITIVITY – DATASET SIZE (all 4 models)
  [saved] sensitivity_03_size_line.png
  [saved] sensitivity_04_size_heatmap.png
  [saved] sensitivity_05_learning_curve_fraud_count.png



,Model,1%,5%,10%,25%,50%,100%
0,LightGBM,0.7636,0.8180,0.8101,0.7517,0.8770,0.9163
1,Logistic Regression,0.7957,0.7363,0.6255,0.8238,0.8260,0.8019
2,Random Forest,0.6797,0.6810,0.7241,0.7576,0.7999,0.8578
3,XGBoost,0.7784,0.7587,0.8537,0.8614,0.8951,0.9150



┌────────────────────────────────────────────────────────────┐
│  📊 Dataset Size Sensitivity                               │
├────────────────────────────────────────────────────────────┤
│  XGBoost gain (5%→100%): +0.1564                           │
│  LightGBM gain (5%→100%): +0.0984                          │
│  LR gain (5%→100%): +0.0656                                │
│  Fastest at full size: LightGBM — scales better than Random Forest│
└────────────────────────────────────────────────────────────┘


## 🌲 8 — Feature Importance

Impurity-based importance is computed for Random Forest, XGBoost, and LightGBM.  
A **consensus heatmap** shows which features appear in the top-15 of multiple models.

> ⚠️ Impurity-based importance can be biased toward high-cardinality features. Results should be interpreted as indicative.

In [11]:
# ============================================================
# 8) FEATURE IMPORTANCE  (RF / XGB / LGBM) + CONSENSUS
# ============================================================
print("\n" + "="*70)
print("8) FEATURE IMPORTANCE")
print("="*70)

fi_models_names = ["Random Forest", "XGBoost", "LightGBM"]
feature_importances: Dict[str, pd.DataFrame] = {}
feature_names = raw_features + pca_features

for mname in fi_models_names:
    model = build_model(mname, SEED, train_prev, raw_features, pca_features)
    if model is None: continue
    model.fit(X_train, y_train)
    clf = model.named_steps["clf"]
    if not hasattr(clf, "feature_importances_"): continue
    fi_df = pd.DataFrame({"feature": feature_names,
                           "importance": clf.feature_importances_})\
              .sort_values("importance", ascending=False).reset_index(drop=True)
    fi_df["rank"] = fi_df.index + 1
    feature_importances[mname] = fi_df
    fi_df.to_csv(RES_DIR / f"phase1_fi_{mname.replace(' ','_')}.csv", index=False)
    print(f"  {mname}: top feature = {fi_df['feature'].iloc[0]} ({fi_df['importance'].iloc[0]:.4f})")

if feature_importances:
    n_fi = len(feature_importances)

    fig, axes = plt.subplots(1, n_fi, figsize=(7*n_fi, 8))
    fig.suptitle("Feature Importance – Top 15 per Model", fontsize=14, fontweight="bold")
    if n_fi == 1: axes = [axes]
    for ax, (mname, fi_df) in zip(axes, feature_importances.items()):
        top15 = fi_df.head(15)
        ax.barh(top15["feature"][::-1], top15["importance"][::-1],
                color=COLORS.get(mname, "steelblue"), edgecolor="black", alpha=0.85)
        ax.set_title(mname, fontsize=12, fontweight="bold")
        ax.set_xlabel("Importance Score")
        for i, (feat, imp) in enumerate(zip(top15["feature"][::-1], top15["importance"][::-1])):
            ax.text(imp + 0.001, i, f"{imp:.4f}", va="center", fontsize=8)
    save_fig("fi_01_top15_bars.png")

    if len(feature_importances) >= 2:
        model_list = list(feature_importances.keys())
        rank_dfs = []
        for mname, fi_df in feature_importances.items():
            tmp = fi_df[["feature","rank"]].copy()
            tmp["model"] = mname
            rank_dfs.append(tmp)
        rank_merged = pd.concat(rank_dfs, ignore_index=True)
        rank_pivot  = rank_merged.pivot(index="feature", columns="model", values="rank")

        if len(model_list) >= 2:
            m1, m2 = model_list[0], model_list[1]
            fig, ax = plt.subplots(figsize=(8, 8))
            valid = rank_pivot.dropna(subset=[m1, m2])
            sc = ax.scatter(valid[m1], valid[m2], s=50, alpha=0.7,
                           c=valid[m1], cmap="viridis")
            top10_m1 = valid.nsmallest(10, m1)
            for feat, row in top10_m1.iterrows():
                ax.annotate(feat, (row[m1], row[m2]),
                            textcoords="offset points", xytext=(5, 3), fontsize=8)
            ax.plot([0, 30], [0, 30], "k--", alpha=0.4, label="Perfect agreement")
            ax.set_xlabel(f"Rank in {m1}", fontsize=11)
            ax.set_ylabel(f"Rank in {m2}", fontsize=11)
            ax.set_title(f"Feature Rank Agreement: {m1} vs {m2}", fontweight="bold")
            ax.legend()
            plt.colorbar(sc, label=f"Rank in {m1}")
            save_fig("fi_02_rank_agreement.png")

    top_n = 15
    first_rank_map = feature_importances[list(feature_importances.keys())[0]].set_index("feature")["rank"].to_dict()
    all_top_feats = sorted(
        set(feat for fi_df in feature_importances.values() for feat in fi_df.head(top_n)["feature"]),
        key=lambda f: first_rank_map.get(f, 99)
    )

    consensus_matrix = pd.DataFrame(index=all_top_feats, columns=list(feature_importances.keys()))
    for mname, fi_df in feature_importances.items():
        rank_map = fi_df.set_index("feature")["rank"].to_dict()
        for feat in all_top_feats:
            consensus_matrix.loc[feat, mname] = rank_map.get(feat, np.nan)
    consensus_matrix = consensus_matrix.astype(float)

    fig, ax = plt.subplots(figsize=(8, 9))
    sns.heatmap(consensus_matrix, annot=True, fmt=".0f", cmap="YlOrRd_r",
                linewidths=0.4, ax=ax, cbar_kws={"label": "Feature Rank (lower=more important)"},
                vmin=1, vmax=top_n)
    ax.set_title(f"Feature Importance Consensus (Rank, Top {top_n} per model)",
                 fontweight="bold")
    ax.set_xlabel("Model"); ax.set_ylabel("Feature")
    save_fig("fi_03_consensus_heatmap.png")

    votes = Counter([f for fi_df in feature_importances.values()
                     for f in fi_df.head(10)["feature"]])
    consensus_df = pd.DataFrame(
        [(f, c) for f, c in votes.items() if c >= 2],
        columns=["feature", "models_voting"]
    ).sort_values("models_voting", ascending=False)
    consensus_df.to_csv(RES_DIR / "phase1_fi_consensus.csv", index=False)
    print("\nConsensus features (top-10 of ≥2 models):")
    print(consensus_df)

# ============================================================
# 9) CALIBRATION ANALYSIS  ← NEW
# Brier Score + ECE + Reliability Diagrams
# (retrofitted from Phase 2 for cross-dataset comparison)
# ============================================================
print("\n" + "="*70)
print("9) CALIBRATION ANALYSIS")
print("="*70)

calib_stats: Dict[str, Dict] = {}
y_test_np = y_test.values

for mname, proba in pr_cache.items():
    brier = brier_score_loss(y_test_np, proba)
    ece   = compute_ece(y_test_np, proba, n_bins=10)
    calib_stats[mname] = {"brier": brier, "ece": ece}
    print(f"  {mname}: Brier={brier:.4f}  ECE={ece:.4f}")

# ── 9a. Reliability diagram + probability histogram ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    "Probability Calibration – Kaggle CC\n"
    "Reliability diagram: does predicted P(fraud) match observed fraud rate?\n"
    "Perfect calibration = diagonal line",
    fontsize=12, fontweight="bold")

ax_cal  = axes[0]
ax_hist = axes[1]

ax_cal.plot([0, 1], [0, 1], "k--", linewidth=1.5, label="Perfect calibration", zorder=0)
for mname, proba in pr_cache.items():
    frac_pos, mean_pred = calibration_curve(y_test_np, proba, n_bins=12, strategy="quantile")
    ece = calib_stats[mname]["ece"]
    ax_cal.plot(mean_pred, frac_pos,
                marker="o", markersize=5, linewidth=2,
                color=COLORS[mname],
                label=f"{mname}  ECE={ece:.3f}")

ax_cal.set_xlabel("Mean Predicted Probability", fontsize=11)
ax_cal.set_ylabel("Observed Fraud Rate", fontsize=11)
ax_cal.set_title("Reliability Diagram", fontsize=11, fontweight="bold")
ax_cal.legend(fontsize=9)
ax_cal.set_xlim(0, 1); ax_cal.set_ylim(0, 1)

for mname, proba in pr_cache.items():
    ax_hist.hist(proba, bins=50, alpha=0.45, color=COLORS[mname], label=mname, density=True)
ax_hist.set_xlabel("Predicted P(fraud)", fontsize=11)
ax_hist.set_ylabel("Density", fontsize=11)
ax_hist.set_title("Distribution of Predicted Probabilities", fontsize=11, fontweight="bold")
ax_hist.legend(fontsize=9)
ax_hist.set_xlim(0, 1)
save_fig("calibration_01_reliability_diagram.png")

# ── 9b. Brier score + ECE bar chart ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    "Calibration Quality Summary – Kaggle CC\n"
    "Brier Score: mean squared error of predicted probabilities (↓ better)\n"
    "ECE: Expected Calibration Error (↓ better)",
    fontsize=11, fontweight="bold")

for ax, (metric_key, ylabel) in zip(axes, [
    ("brier", "Brier Score"),
    ("ece",   "Expected Calibration Error (ECE)")
]):
    names  = [m for m in MODEL_NAMES if m in calib_stats]
    values = [calib_stats[m][metric_key] for m in names]
    colors = [COLORS[m] for m in names]
    bars = ax.bar(names, values, color=colors, edgecolor="black", alpha=0.85)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(ylabel, fontsize=10, fontweight="bold")
    ax.tick_params(axis="x", rotation=15)
    ax.set_ylim(0, max(values) * 1.35)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.02,
                f"{val:.4f}", ha="center", fontsize=9, fontweight="bold")

    if metric_key == "brier":
        naive_brier = brier_score_loss(y_test_np,
                                       np.full_like(y_test_np, y_test_np.mean(), dtype=float))
        ax.axhline(naive_brier, color="gray", linestyle="--", linewidth=1.5,
                   label=f"Naive baseline: {naive_brier:.4f}")
        ax.legend(fontsize=8)

save_fig("calibration_02_scores.png")

pd.DataFrame([
    {"model": m, "brier_score": v["brier"], "ece": v["ece"]}
    for m, v in calib_stats.items()
]).to_csv(RES_DIR / "phase1_calibration_stats.csv", index=False)
print("  Calibration stats saved.")

# ============================================================
# 10) STATISTICAL SIGNIFICANCE TESTS  ← NEW
# Paired t-tests and Wilcoxon tests on per-repeat PR-AUC
# ============================================================
print("\n" + "="*70)
print("10) STATISTICAL SIGNIFICANCE TESTS")
print(f"    Metric: PR-AUC across {N_REPEATS} repeats")
print("="*70)

from itertools import combinations

available_models = [m for m in MODEL_NAMES if m in repeats_df["model"].unique()]
sig_rows = []

for m1, m2 in combinations(available_models, 2):
    vals_m1 = repeats_df[repeats_df["model"]==m1].sort_values("repeat")["pr_auc"].values
    vals_m2 = repeats_df[repeats_df["model"]==m2].sort_values("repeat")["pr_auc"].values

    # Paired t-test (assumes normality; reasonable for N>=5)
    if len(vals_m1) >= 2 and len(vals_m2) >= 2:
        n = min(len(vals_m1), len(vals_m2))
        t_stat, t_pval = ttest_rel(vals_m1[:n], vals_m2[:n])

        # Wilcoxon signed-rank (non-parametric; safer for small N)
        try:
            w_stat, w_pval = wilcoxon(vals_m1[:n], vals_m2[:n])
        except Exception:
            w_stat, w_pval = np.nan, np.nan

        mean_diff = float(np.mean(vals_m1[:n]) - np.mean(vals_m2[:n]))
        sig_rows.append({
            "model_A": m1, "model_B": m2,
            "mean_A": float(np.mean(vals_m1[:n])),
            "mean_B": float(np.mean(vals_m2[:n])),
            "mean_diff (A-B)": mean_diff,
            "t_statistic": float(t_stat),
            "t_pvalue": float(t_pval),
            "wilcoxon_stat": float(w_stat) if not np.isnan(w_stat) else None,
            "wilcoxon_pvalue": float(w_pval) if not np.isnan(w_pval) else None,
            "significant_p05": bool(t_pval < 0.05),
        })
        marker = "*" if t_pval < 0.05 else " "
        print(f"  {m1} vs {m2}: diff={mean_diff:+.4f}  "
              f"t-p={t_pval:.4f}  W-p={w_pval:.4f}  {marker}")

sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(RES_DIR / "phase1_significance_tests.csv", index=False)

# ── Significance heatmap ─────────────────────────────────────
n_m = len(available_models)
pval_matrix = pd.DataFrame(np.ones((n_m, n_m)),
                            index=available_models, columns=available_models)
for row in sig_rows:
    pval_matrix.loc[row["model_A"], row["model_B"]] = row["t_pvalue"]
    pval_matrix.loc[row["model_B"], row["model_A"]] = row["t_pvalue"]

# fill diagonal safely via an explicit numpy copy
arr = pval_matrix.values.copy()
np.fill_diagonal(arr, 1.0)
pval_matrix = pd.DataFrame(arr, index=available_models, columns=available_models)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(pval_matrix.astype(float), annot=True, fmt=".3f",
            cmap="RdYlGn_r", ax=ax, vmin=0, vmax=0.1,
            linewidths=0.5, cbar_kws={"label": "Paired t-test p-value"})
ax.set_title(
    f"Pairwise Significance Tests – PR-AUC ({N_REPEATS} repeats)\n"
    "Green = significant difference (p < 0.05), Red = not significant\n"
    f"Note: N={N_REPEATS} repeats; interpret with caution (low power)",
    fontsize=10, fontweight="bold")
save_fig("significance_01_pairwise_heatmap.png")
print("  Significance test results saved.")

# ── Consensus table ───────────────────────────────────────────────
if 'consensus_df' in dir() and not consensus_df.empty:
    styled_table(consensus_df.head(10),
                 higher_is_better=["models_voting"],
                 title="Feature Importance Consensus — Top features appearing in ≥2 models' top-10")
    print_summary("Feature Importance", [
        ("Top consensus feature", consensus_df.iloc[0]['feature'] if not consensus_df.empty else "N/A"),
        ("Features in all 3 models' top-10", str((consensus_df['models_voting']==3).sum())),
        ("Features in 2+ models' top-10", str(len(consensus_df))),
    ])



8) FEATURE IMPORTANCE
  Random Forest: top feature = V14 (0.2164)
  XGBoost: top feature = V14 (0.3283)
  LightGBM: top feature = V14 (321.0000)
  [saved] fi_01_top15_bars.png
  [saved] fi_02_rank_agreement.png
  [saved] fi_03_consensus_heatmap.png

Consensus features (top-10 of ≥2 models):
  feature  models_voting
0     V14              3
1     V12              3
2      V4              3
3     V10              3
4     V17              2
5     V11              2
6     V13              2
7     V19              2

9) CALIBRATION ANALYSIS
  Logistic Regression: Brier=0.0228  ECE=0.0715
  Random Forest: Brier=0.0012  ECE=0.0111
  XGBoost: Brier=0.0010  ECE=0.0029
  LightGBM: Brier=0.0007  ECE=0.0014
  [saved] calibration_01_reliability_diagram.png
  [saved] calibration_02_scores.png
  Calibration stats saved.

10) STATISTICAL SIGNIFICANCE TESTS
    Metric: PR-AUC across 5 repeats
  Logistic Regression vs Random Forest: diff=-0.0514  t-p=0.0143  W-p=0.0625  *
  Logistic Regression vs XGBoo

,feature,models_voting
0,V14,3
1,V12,3
2,V4,3
3,V10,3
4,V17,2
5,V11,2
6,V13,2
7,V19,2



┌────────────────────────────────────────────────────────────┐
│  📊 Feature Importance                                     │
├────────────────────────────────────────────────────────────┤
│  Top consensus feature: V14                                │
│  Features in all 3 models' top-10: 4                       │
│  Features in 2+ models' top-10: 8                          │
└────────────────────────────────────────────────────────────┘


## 📐 9 — Probability Calibration

In fraud detection, **probability calibration matters** for operational decisions:
- A well-calibrated model with P(fraud)=0.8 means ~80% of such cases are truly fraud
- Poor calibration leads to suboptimal threshold selection and misleading risk scores

**Metrics:**
- **Brier Score** — mean squared error of predicted probabilities (↓ better)
- **ECE** — Expected Calibration Error, average gap between predicted probability and observed rate (↓ better)

In [12]:
# ============================================================
# 9) CALIBRATION ANALYSIS  ← NEW
# Brier Score + ECE + Reliability Diagrams
# (retrofitted from Phase 2 for cross-dataset comparison)
# ============================================================
print("\n" + "="*70)
print("9) CALIBRATION ANALYSIS")
print("="*70)

calib_stats: Dict[str, Dict] = {}
y_test_np = y_test.values

for mname, proba in pr_cache.items():
    brier = brier_score_loss(y_test_np, proba)
    ece   = compute_ece(y_test_np, proba, n_bins=10)
    calib_stats[mname] = {"brier": brier, "ece": ece}
    print(f"  {mname}: Brier={brier:.4f}  ECE={ece:.4f}")

# ── 9a. Reliability diagram + probability histogram ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    "Probability Calibration – Kaggle CC\n"
    "Reliability diagram: does predicted P(fraud) match observed fraud rate?\n"
    "Perfect calibration = diagonal line",
    fontsize=12, fontweight="bold")

ax_cal  = axes[0]
ax_hist = axes[1]

ax_cal.plot([0, 1], [0, 1], "k--", linewidth=1.5, label="Perfect calibration", zorder=0)
for mname, proba in pr_cache.items():
    frac_pos, mean_pred = calibration_curve(y_test_np, proba, n_bins=12, strategy="quantile")
    ece = calib_stats[mname]["ece"]
    ax_cal.plot(mean_pred, frac_pos,
                marker="o", markersize=5, linewidth=2,
                color=COLORS[mname],
                label=f"{mname}  ECE={ece:.3f}")

ax_cal.set_xlabel("Mean Predicted Probability", fontsize=11)
ax_cal.set_ylabel("Observed Fraud Rate", fontsize=11)
ax_cal.set_title("Reliability Diagram", fontsize=11, fontweight="bold")
ax_cal.legend(fontsize=9)
ax_cal.set_xlim(0, 1); ax_cal.set_ylim(0, 1)

for mname, proba in pr_cache.items():
    ax_hist.hist(proba, bins=50, alpha=0.45, color=COLORS[mname], label=mname, density=True)
ax_hist.set_xlabel("Predicted P(fraud)", fontsize=11)
ax_hist.set_ylabel("Density", fontsize=11)
ax_hist.set_title("Distribution of Predicted Probabilities", fontsize=11, fontweight="bold")
ax_hist.legend(fontsize=9)
ax_hist.set_xlim(0, 1)
save_fig("calibration_01_reliability_diagram.png")

# ── 9b. Brier score + ECE bar chart ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    "Calibration Quality Summary – Kaggle CC\n"
    "Brier Score: mean squared error of predicted probabilities (↓ better)\n"
    "ECE: Expected Calibration Error (↓ better)",
    fontsize=11, fontweight="bold")

for ax, (metric_key, ylabel) in zip(axes, [
    ("brier", "Brier Score"),
    ("ece",   "Expected Calibration Error (ECE)")
]):
    names  = [m for m in MODEL_NAMES if m in calib_stats]
    values = [calib_stats[m][metric_key] for m in names]
    colors = [COLORS[m] for m in names]
    bars = ax.bar(names, values, color=colors, edgecolor="black", alpha=0.85)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(ylabel, fontsize=10, fontweight="bold")
    ax.tick_params(axis="x", rotation=15)
    ax.set_ylim(0, max(values) * 1.35)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.02,
                f"{val:.4f}", ha="center", fontsize=9, fontweight="bold")

    if metric_key == "brier":
        naive_brier = brier_score_loss(y_test_np,
                                       np.full_like(y_test_np, y_test_np.mean(), dtype=float))
        ax.axhline(naive_brier, color="gray", linestyle="--", linewidth=1.5,
                   label=f"Naive baseline: {naive_brier:.4f}")
        ax.legend(fontsize=8)

save_fig("calibration_02_scores.png")

pd.DataFrame([
    {"model": m, "brier_score": v["brier"], "ece": v["ece"]}
    for m, v in calib_stats.items()
]).to_csv(RES_DIR / "phase1_calibration_stats.csv", index=False)
print("  Calibration stats saved.")


# ── Calibration summary table ─────────────────────────────────────
calib_df = pd.DataFrame([
    {"Model": m, "Brier Score": v["brier"], "ECE": v["ece"]}
    for m, v in calib_stats.items()
]).sort_values("Brier Score")
styled_table(calib_df,
             higher_is_better=[],
             title="Calibration Quality — Lower is better for both metrics")

best_cal = calib_df.iloc[0]
worst_cal = calib_df.iloc[-1]
print_summary("Calibration Analysis", [
    ("Best calibrated model", f"{best_cal['Model']}  Brier={best_cal['Brier Score']:.4f}  ECE={best_cal['ECE']:.4f}"),
    ("Worst calibrated model", f"{worst_cal['Model']}  Brier={worst_cal['Brier Score']:.4f}  ECE={worst_cal['ECE']:.4f}"),
    ("Naive baseline Brier", f"{brier_score_loss(y_test_np, np.full_like(y_test_np, y_test_np.mean(), dtype=float)):.4f}"),
    ("Interpretation", "LightGBM produces most reliable probability scores"),
])



9) CALIBRATION ANALYSIS
  Logistic Regression: Brier=0.0228  ECE=0.0715
  Random Forest: Brier=0.0012  ECE=0.0111
  XGBoost: Brier=0.0010  ECE=0.0029
  LightGBM: Brier=0.0007  ECE=0.0014
  [saved] calibration_01_reliability_diagram.png
  [saved] calibration_02_scores.png
  Calibration stats saved.



,Model,Brier Score,ECE
3,LightGBM,0.0007,0.0014
2,XGBoost,0.0010,0.0029
1,Random Forest,0.0012,0.0111
0,Logistic Regression,0.0228,0.0715



┌────────────────────────────────────────────────────────────┐
│  📊 Calibration Analysis                                   │
├────────────────────────────────────────────────────────────┤
│  Best calibrated model: LightGBM  Brier=0.0007  ECE=0.0014 │
│  Worst calibrated model: Logistic Regression  Brier=0.0228  ECE=0.0715│
│  Naive baseline Brier: 0.0017                              │
│  Interpretation: LightGBM produces most reliable probability scores│
└────────────────────────────────────────────────────────────┘


## 📊 10 — Statistical Significance Tests

Pairwise comparisons of PR-AUC across **5 repeated experiments** using:
- **Paired t-test** — parametric test assuming normally distributed differences
- **Wilcoxon signed-rank** — non-parametric alternative, safer for small N

> ⚠️ With N=5 repeats, statistical power is low. Significant results (p<0.05) should be interpreted as **indicative** rather than conclusive. Increasing N_REPEATS to 10+ would yield more reliable inferences.

In [13]:
# ============================================================
# 10) STATISTICAL SIGNIFICANCE TESTS
# Paired t-tests and Wilcoxon tests on per-repeat PR-AUC
# ============================================================
print("\n" + "="*70)
print("10) STATISTICAL SIGNIFICANCE TESTS")
print(f"    Metric: PR-AUC across {N_REPEATS} repeats")
print("="*70)

from itertools import combinations

available_models = [m for m in MODEL_NAMES if m in repeats_df["model"].unique()]
sig_rows = []

for m1, m2 in combinations(available_models, 2):
    vals_m1 = repeats_df[repeats_df["model"]==m1].sort_values("repeat")["pr_auc"].values
    vals_m2 = repeats_df[repeats_df["model"]==m2].sort_values("repeat")["pr_auc"].values

    if len(vals_m1) >= 2 and len(vals_m2) >= 2:
        n = min(len(vals_m1), len(vals_m2))
        t_stat, t_pval = ttest_rel(vals_m1[:n], vals_m2[:n])

        try:
            w_stat, w_pval = wilcoxon(vals_m1[:n], vals_m2[:n])
        except Exception:
            w_stat, w_pval = np.nan, np.nan

        mean_diff = float(np.mean(vals_m1[:n]) - np.mean(vals_m2[:n]))
        sig_rows.append({
            "model_A": m1, "model_B": m2,
            "mean_A": float(np.mean(vals_m1[:n])),
            "mean_B": float(np.mean(vals_m2[:n])),
            "mean_diff (A-B)": mean_diff,
            "t_statistic": float(t_stat),
            "t_pvalue": float(t_pval),
            "wilcoxon_stat": float(w_stat) if not np.isnan(w_stat) else None,
            "wilcoxon_pvalue": float(w_pval) if not np.isnan(w_pval) else None,
            "significant_p05": bool(t_pval < 0.05),
        })
        marker = "*" if t_pval < 0.05 else " "
        print(f"  {m1} vs {m2}: diff={mean_diff:+.4f}  "
              f"t-p={t_pval:.4f}  W-p={w_pval:.4f}  {marker}")

sig_df = pd.DataFrame(sig_rows)
# ── Bonferroni correction ─────────────────────────────────────
# With C(4,2)=6 pairwise comparisons, the familywise error rate
# at α=0.05 inflates. Bonferroni-corrected threshold: α/6 ≈ 0.0083.
n_comparisons = len(sig_rows)
alpha_bonferroni = 0.05 / max(n_comparisons, 1)
print(f"\n  Bonferroni-corrected α: 0.05 / {n_comparisons} = {alpha_bonferroni:.4f}")
print(f"  Pairs significant after Bonferroni: "
      f"{sum(r['t_pvalue'] < alpha_bonferroni for r in sig_rows)} / {n_comparisons}")
print(f"  Note: N={N_REPEATS} repeats — interpret as indicative, not conclusive.")
if not sig_df.empty:
    sig_df["bonferroni_sig"] = sig_df["t_pvalue"] < alpha_bonferroni
sig_df.to_csv(RES_DIR / "phase1_significance_tests.csv", index=False)

# ── Significance heatmap ─────────────────────────────────────
n_m = len(available_models)
pval_matrix = pd.DataFrame(np.ones((n_m, n_m)),
                            index=available_models, columns=available_models)
for row in sig_rows:
    pval_matrix.loc[row["model_A"], row["model_B"]] = row["t_pvalue"]
    pval_matrix.loc[row["model_B"], row["model_A"]] = row["t_pvalue"]

# FIX: extract a writable numpy copy before fill_diagonal
arr = pval_matrix.values.copy()
np.fill_diagonal(arr, 1.0)
pval_matrix = pd.DataFrame(arr, index=available_models, columns=available_models)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(pval_matrix.astype(float), annot=True, fmt=".3f",
            cmap="RdYlGn_r", ax=ax, vmin=0, vmax=0.1,
            linewidths=0.5, cbar_kws={"label": "Paired t-test p-value"})
ax.set_title(
    f"Pairwise Significance Tests – PR-AUC ({N_REPEATS} repeats)\n"
    "Green = significant difference (p < 0.05), Red = not significant\n"
    f"Note: N={N_REPEATS} repeats; interpret with caution (low power)",
    fontsize=10, fontweight="bold")
save_fig("significance_01_pairwise_heatmap.png")
print("  Significance test results saved.")

# ── Significance table ────────────────────────────────────────────
if not sig_df.empty:
    sig_display = sig_df[["model_A","model_B","mean_A","mean_B",
                           "mean_diff (A-B)","t_pvalue","wilcoxon_pvalue","significant_p05"]].copy()
    sig_display.columns = ["Model A","Model B","Mean A","Mean B",
                            "Diff (A-B)","t p-value","Wilcoxon p-value","Sig. p<0.05"]
    styled_table(sig_display,
                 title="Pairwise Statistical Tests — PR-AUC over 5 repeats")

    n_sig = sig_df["significant_p05"].sum()

    # FIX: dynamic per-pair summary rows instead of hardcoded model names
    summary_rows = [("Significant pairs (p<0.05)", f"{n_sig} / {len(sig_df)}")]
    for _, r in sig_df.iterrows():
        summary_rows.append(
            (f"{r['model_A']} vs {r['model_B']} p-value", f"{r['t_pvalue']:.4f}")
        )
    summary_rows.append(("Key finding", "See p-values above; pairs marked * are significant at p<0.05"))

    print_summary("Significance Tests", summary_rows)


10) STATISTICAL SIGNIFICANCE TESTS
    Metric: PR-AUC across 5 repeats
  Logistic Regression vs Random Forest: diff=-0.0514  t-p=0.0143  W-p=0.0625  *
  Logistic Regression vs XGBoost: diff=-0.1003  t-p=0.0118  W-p=0.0625  *
  Logistic Regression vs LightGBM: diff=-0.0979  t-p=0.0064  W-p=0.0625  *
  Random Forest vs XGBoost: diff=-0.0488  t-p=0.0429  W-p=0.1250  *
  Random Forest vs LightGBM: diff=-0.0465  t-p=0.0104  W-p=0.0625  *
  XGBoost vs LightGBM: diff=+0.0024  t-p=0.8499  W-p=0.6250   

  Bonferroni-corrected α: 0.05 / 6 = 0.0083
  Pairs significant after Bonferroni: 1 / 6
  Note: N=5 repeats — interpret as indicative, not conclusive.
  [saved] significance_01_pairwise_heatmap.png
  Significance test results saved.



,Model A,Model B,Mean A,Mean B,Diff (A-B),t p-value,Wilcoxon p-value,Sig. p<0.05
0,Logistic Regression,Random Forest,0.7326,0.7841,-0.0514,0.0143,0.0625,True
1,Logistic Regression,XGBoost,0.7326,0.8329,-0.1003,0.0118,0.0625,True
2,Logistic Regression,LightGBM,0.7326,0.8305,-0.0979,0.0064,0.0625,True
3,Random Forest,XGBoost,0.7841,0.8329,-0.0488,0.0429,0.1250,True
4,Random Forest,LightGBM,0.7841,0.8305,-0.0465,0.0104,0.0625,True
5,XGBoost,LightGBM,0.8329,0.8305,0.0024,0.8499,0.6250,False



┌────────────────────────────────────────────────────────────┐
│  📊 Significance Tests                                     │
├────────────────────────────────────────────────────────────┤
│  Significant pairs (p<0.05): 5 / 6                         │
│  Logistic Regression vs Random Forest p-value: 0.0143      │
│  Logistic Regression vs XGBoost p-value: 0.0118            │
│  Logistic Regression vs LightGBM p-value: 0.0064           │
│  Random Forest vs XGBoost p-value: 0.0429                  │
│  Random Forest vs LightGBM p-value: 0.0104                 │
│  XGBoost vs LightGBM p-value: 0.8499                       │
│  Key finding: See p-values above; pairs marked * are significant at p<0.05│
└────────────────────────────────────────────────────────────┘


## 🔍 11 — SHAP Explainability Analysis

SHAP (SHapley Additive exPlanations) values are computed for **LightGBM** — the best or tied-best model across all phases.

Unlike impurity-based importance, SHAP quantifies each feature's **directional contribution** to individual predictions, grounded in cooperative game theory.

**Three outputs:**
- **Beeswarm plot** — global feature importance with direction (red = high value pushing toward fraud, blue = low value)
- **Bar plot** — mean |SHAP| per feature (magnitude-only global ranking)
- **Dependence plot** — how the most important feature's SHAP value varies with its raw value

> SHAP values are computed on a stratified subsample of the test set (up to 2,000 rows) for computational efficiency.

In [14]:
# ============================================================
# 11) SHAP EXPLAINABILITY — LightGBM (Phase 1)
# ============================================================
print("\n" + "="*70)
print("11) SHAP ANALYSIS — LightGBM")
print("="*70)

try:
    import shap
    print("SHAP version:", shap.__version__)
    shap_available = True
except Exception as e:
    print("WARNING: SHAP import failed:", e)
    shap_available = False

if shap_available and "LightGBM" in model_cache:
    lgbm_pipeline = model_cache["LightGBM"]
    clf_shap      = lgbm_pipeline.named_steps["clf"]
    prep_shap     = lgbm_pipeline.named_steps["prep"]

    # Preprocessed test data
    Xp_test_shap = prep_shap.transform(X_test)
    if hasattr(Xp_test_shap, "toarray"):
        Xp_test_shap = Xp_test_shap.toarray()

    feature_names_shap = raw_features + pca_features

    # Subsample for speed
    np.random.seed(SEED)
    n_shap = min(2000, len(X_test))
    idx_shap = np.random.choice(len(X_test), n_shap, replace=False)
    Xp_shap_sample = Xp_test_shap[idx_shap]
    y_shap_sample  = y_test.values[idx_shap]

    print(f"  Computing SHAP values on {n_shap} test samples...")
    explainer = shap.TreeExplainer(clf_shap)
    shap_values = explainer.shap_values(Xp_shap_sample)

    if isinstance(shap_values, list):
        sv = shap_values[1]
    else:
        sv = shap_values

    import pandas as pd
    Xp_shap_df = pd.DataFrame(Xp_shap_sample, columns=feature_names_shap)

    # 11a. Beeswarm summary plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(sv, Xp_shap_df, max_display=15, show=False)
    plt.title("SHAP Beeswarm — LightGBM (Phase 1: Kaggle CC)\n"
              "Red = high feature value pushes toward fraud | Blue = low value",
              fontsize=12, fontweight="bold")
    plt.tight_layout()
    save_fig("shap_01_beeswarm.png")
    print("  [saved] shap_01_beeswarm.png")

    # 11b. Bar plot
    plt.figure(figsize=(10, 7))
    shap.summary_plot(sv, Xp_shap_df, plot_type="bar", max_display=15, show=False)
    plt.title("SHAP Feature Importance (mean |SHAP|) — LightGBM (Phase 1)",
              fontsize=12, fontweight="bold")
    plt.tight_layout()
    save_fig("shap_02_bar.png")
    print("  [saved] shap_02_bar.png")

    # 11c. Dependence plot — top feature
    mean_abs_shap = np.abs(sv).mean(axis=0)
    top_feat_idx  = int(np.argmax(mean_abs_shap))
    top_feat_name = feature_names_shap[top_feat_idx]
    print(f"  Top SHAP feature: {top_feat_name}")

    plt.figure(figsize=(9, 6))
    shap.dependence_plot(top_feat_idx, sv, Xp_shap_df,
                         feature_names=feature_names_shap, show=False)
    plt.title(f"SHAP Dependence Plot — {top_feat_name} (Phase 1)\n"
              "Shows how this feature's value drives fraud probability",
              fontsize=11, fontweight="bold")
    plt.tight_layout()
    save_fig("shap_03_dependence.png")
    print("  [saved] shap_03_dependence.png")

    # 11d. Save SHAP summary table
    shap_importance = pd.DataFrame({
        "feature": feature_names_shap,
        "mean_abs_shap": mean_abs_shap,
        "mean_shap": sv.mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    shap_importance["rank"] = shap_importance.index + 1
    shap_importance.to_csv(RES_DIR / "phase1_shap_importance.csv", index=False)

    print("\nTop 10 features by mean |SHAP|:")
    print(shap_importance[["rank", "feature", "mean_abs_shap", "mean_shap"]]
          .head(10).to_string(index=False))

    print_summary("SHAP Analysis (LightGBM — Phase 1)", [
        ("Top SHAP feature", top_feat_name),
        ("Mean |SHAP| top feature", f"{mean_abs_shap[top_feat_idx]:.4f}"),
        ("Samples used", f"{n_shap}"),
        ("Outputs", "shap_01_beeswarm.png / shap_02_bar.png / shap_03_dependence.png"),
    ])
else:
    if not shap_available:
        print("  SHAP not available — skipping section.")
    else:
        print("  LightGBM model not in cache — skipping SHAP section.")


11) SHAP ANALYSIS — LightGBM
SHAP version: 0.46.0
  Computing SHAP values on 2000 test samples...
  [saved] shap_01_beeswarm.png
  [saved] shap_01_beeswarm.png
  [saved] shap_02_bar.png
  [saved] shap_02_bar.png
  Top SHAP feature: V14
  [saved] shap_03_dependence.png
  [saved] shap_03_dependence.png

Top 10 features by mean |SHAP|:
 rank feature  mean_abs_shap  mean_shap
    1     V14       0.283273  -0.025774
    2      V4       0.242488  -0.011859
    3     V12       0.147171   0.000086
    4     V26       0.088450   0.008314
    5     V11       0.080221   0.006413
    6      V8       0.076343  -0.004705
    7      V3       0.066361  -0.005361
    8     V19       0.058632  -0.002316
    9      V5       0.056505  -0.005007
   10     V20       0.056012   0.000641

┌────────────────────────────────────────────────────────────┐
│  📊 SHAP Analysis (LightGBM — Phase 1)                     │
├────────────────────────────────────────────────────────────┤
│  Top SHAP feature: V14           

## 📋 12 — Summary Dashboard

One-page visual overview of all Phase 1 results.

In [15]:
# ============================================================
# 12) SUMMARY DASHBOARD  (updated: calibration panel added)
# ============================================================
print("\n" + "="*70)
print("12) SUMMARY DASHBOARD")
print("="*70)

fig = plt.figure(figsize=(20, 26))
fig.suptitle("GP Phase 1 – Summary Dashboard\nDataset: Kaggle Credit Card Fraud",
             fontsize=16, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(5, 3, figure=fig, hspace=0.50, wspace=0.35)

# ── Row 0: Class distribution + config + time ────────────────
ax_pie  = fig.add_subplot(gs[0, 0])
ax_text = fig.add_subplot(gs[0, 1])
ax_time = fig.add_subplot(gs[0, 2])

counts_vals = [y.value_counts().get(0, 0), y.value_counts().get(1, 0)]
ax_pie.pie(counts_vals, labels=["Non-Fraud", "Fraud"],
           colors=["#4C72B0", "#C44E52"],
           autopct=lambda p: f"{p:.3f}%", explode=[0, 0.12], startangle=90)
ax_pie.set_title("Class Distribution", fontweight="bold")

info_text = (
    f"Dataset: Kaggle Credit Card Fraud\n"
    f"Total rows : {len(df):,}\n"
    f"Features   : {len(FEATURES)} (Time, Amount, V1-V28)\n"
    f"Fraud cases: {y.sum()} ({y.mean():.4%})\n"
    f"Imbalance  : {y.value_counts()[0]//y.sum():,}:1\n\n"
    f"Split: 70% Train / 15% Val / 15% Test\n"
    f"Repeats: N={N_REPEATS}  Bootstrap: N={N_BOOTSTRAP}\n"
    f"Threshold: Max-F1 on Validation\n"
    f"Imbalance strategy: class_weight=balanced"
)
ax_text.text(0.05, 0.5, info_text, transform=ax_text.transAxes,
             fontsize=10, verticalalignment="center",
             bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
ax_text.axis("off")
ax_text.set_title("Experiment Configuration", fontweight="bold")

if "TimeHours" in df.columns:
    ax_time.hist(df.loc[df[TARGET_COL]==0, "HourOfDay"], bins=24,
                 alpha=0.6, color="#4C72B0", label="Non-Fraud", density=True)
    ax_time.hist(df.loc[df[TARGET_COL]==1, "HourOfDay"], bins=24,
                 alpha=0.6, color="#C44E52", label="Fraud", density=True)
    ax_time.set_xlabel("Hour of Day"); ax_time.set_ylabel("Density")
    ax_time.set_title("Transaction Hour by Class", fontweight="bold")
    ax_time.legend(fontsize=8)
else:
    ax_time.axis("off")

# ── Row 1: PR curves + ROC curves + best model CM ────────────
ax_pr  = fig.add_subplot(gs[1, 0])
ax_roc = fig.add_subplot(gs[1, 1])
ax_cm  = fig.add_subplot(gs[1, 2])

ax_pr.hlines(float(y_test.mean()), 0, 1, linestyles="--", colors="gray", linewidth=1.2)
for mname, proba in pr_cache.items():
    p, r, _ = precision_recall_curve(y_test.values, proba, pos_label=FRAUD_LABEL)
    ap = average_precision_score(y_test.values, proba)
    ax_pr.plot(r, p, color=COLORS[mname], linewidth=1.5, label=f"{mname[:2]}. AP={ap:.3f}")
ax_pr.set_xlabel("Recall"); ax_pr.set_ylabel("Precision")
ax_pr.set_title("PR Curves (Test)", fontweight="bold")
ax_pr.legend(fontsize=7); ax_pr.set_xlim(0,1); ax_pr.set_ylim(0,1)

ax_roc.plot([0,1],[0,1],"k--", linewidth=1)
for mname, proba in pr_cache.items():
    fp_r, tp_r, _ = roc_curve(y_test.values, proba, pos_label=FRAUD_LABEL)
    roc = roc_auc_score(y_test.values, proba)
    ax_roc.plot(fp_r, tp_r, color=COLORS[mname], linewidth=1.5, label=f"{mname[:2]}. {roc:.3f}")
ax_roc.set_xlabel("FPR"); ax_roc.set_ylabel("TPR")
ax_roc.set_title("ROC Curves (Test)", fontweight="bold")
ax_roc.legend(fontsize=7)

best_model_name = baseline_df.iloc[0]["model"]
proba_best = pr_cache.get(best_model_name)
if proba_best is not None:
    thr_best = thr_cache[best_model_name]
    y_pred_best = (proba_best >= thr_best).astype(int)
    cm_best = confusion_matrix(y_test.values, y_pred_best, labels=[NORMAL_LABEL, FRAUD_LABEL])
    sns.heatmap(cm_best, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Non-Fraud","Fraud"],
                yticklabels=["Non-Fraud","Fraud"],
                cbar=False, ax=ax_cm, annot_kws={"size":11})
    ax_cm.set_xlabel("Predicted"); ax_cm.set_ylabel("Actual")
    ax_cm.set_title(f"Best Model CM\n({best_model_name})", fontweight="bold")

# ── Row 2: Performance bars + Calibration ────────────────────
ax_bar  = fig.add_subplot(gs[2, 0:2])
ax_calib = fig.add_subplot(gs[2, 2])

metrics_show = ["pr_auc", "ap", "f1", "recall", "precision"]
x = np.arange(len(metrics_show))
w = 0.8 / max(len(baseline_df), 1)
for i, (_, row) in enumerate(baseline_df.iterrows()):
    vals = [row.get(m, 0) for m in metrics_show]
    offset = (i - len(baseline_df)/2 + 0.5) * w
    ax_bar.bar(x + offset, vals, width=w*0.85,
               color=COLORS.get(row["model"], "steelblue"),
               edgecolor="black", alpha=0.85, label=row["model"])
ax_bar.set_xticks(x)
ax_bar.set_xticklabels(["PR-AUC", "Avg Prec", "F1", "Recall", "Precision"], fontsize=10)
ax_bar.set_ylim(0, 1.15)
ax_bar.set_ylabel("Score")
ax_bar.set_title("Model Performance Comparison (Test Set)", fontweight="bold")
ax_bar.legend(fontsize=8, loc="upper right")

# Mini calibration: Brier score comparison
if calib_stats:
    calib_names = [m for m in MODEL_NAMES if m in calib_stats]
    brier_vals  = [calib_stats[m]["brier"] for m in calib_names]
    ax_calib.bar(calib_names, brier_vals,
                 color=[COLORS.get(m, "gray") for m in calib_names],
                 edgecolor="black", alpha=0.85)
    ax_calib.set_title("Brier Score (↓ better)", fontweight="bold")
    ax_calib.set_ylabel("Brier Score")
    ax_calib.tick_params(axis="x", rotation=20)
    for i, (name, val) in enumerate(zip(calib_names, brier_vals)):
        ax_calib.text(i, val + max(brier_vals)*0.02, f"{val:.4f}",
                      ha="center", fontsize=8, fontweight="bold")

# ── Row 3: Imbalance sensitivity + Size sensitivity ──────────
ax_imb = fig.add_subplot(gs[3, 0])
ax_sz  = fig.add_subplot(gs[3, 1])
ax_fi  = fig.add_subplot(gs[3, 2])

for mname in MODEL_NAMES:
    sub = imb_df[imb_df["model"]==mname].sort_values("target_ratio")
    if sub.empty: continue
    ax_imb.plot(sub["target_ratio"], sub["pr_auc"],
                marker=MARKERS.get(mname,"o"), color=COLORS[mname],
                linewidth=1.5, markersize=5, label=mname[:5])
ax_imb.set_xscale("log"); ax_imb.set_xlabel("Fraud Prevalence (log)")
ax_imb.set_ylabel("PR-AUC"); ax_imb.set_ylim(0, 1)
ax_imb.set_title("Imbalance Sensitivity", fontweight="bold")
ax_imb.legend(fontsize=7)

for mname in MODEL_NAMES:
    sub = size_df[size_df["model"]==mname].sort_values("train_fraction")
    if sub.empty: continue
    ax_sz.plot(sub["train_fraction"], sub["pr_auc"],
               marker=MARKERS.get(mname,"o"), color=COLORS[mname],
               linewidth=1.5, markersize=5, label=mname[:5])
ax_sz.set_xlabel("Training Fraction"); ax_sz.set_ylabel("PR-AUC")
ax_sz.set_title("Size Sensitivity", fontweight="bold")
ax_sz.legend(fontsize=7); ax_sz.set_ylim(0, 1)

if feature_importances:
    fi_best_name = [m for m in fi_models_names if m in feature_importances]
    if fi_best_name:
        fi_show = feature_importances[fi_best_name[0]].head(10)
        ax_fi.barh(fi_show["feature"][::-1], fi_show["importance"][::-1],
                   color=COLORS.get(fi_best_name[0], "steelblue"),
                   edgecolor="black", alpha=0.85)
        ax_fi.set_xlabel("Importance")
        ax_fi.set_title(f"Top 10 Features\n({fi_best_name[0]})", fontweight="bold")
else:
    ax_fi.axis("off")

# ── Row 4: Timing + significance heatmap ─────────────────────
ax_tim = fig.add_subplot(gs[4, 0])
ax_sig = fig.add_subplot(gs[4, 1:3])

sub_tim = baseline_df.set_index("model").reindex(
    [m for m in MODEL_NAMES if m in baseline_df["model"].values])
ax_tim.barh(sub_tim.index, sub_tim["train_time_s"],
            color=[COLORS.get(m, "gray") for m in sub_tim.index],
            edgecolor="black", alpha=0.85)
ax_tim.set_xlabel("Training Time (s)")
ax_tim.set_title("Training Time Comparison", fontweight="bold")
for i, (idx, row) in enumerate(sub_tim.iterrows()):
    ax_tim.text(row["train_time_s"]*1.02, i, f"{row['train_time_s']:.1f}s",
                va="center", fontsize=9)

if not pval_matrix.empty:
    sns.heatmap(pval_matrix.astype(float), annot=True, fmt=".3f",
                cmap="RdYlGn_r", ax=ax_sig, vmin=0, vmax=0.1,
                linewidths=0.5, cbar_kws={"label": "p-value"})
    ax_sig.set_title(
        f"Pairwise Significance (paired t-test, N={N_REPEATS} repeats)\n"
        "Green = significant (p<0.05)",
        fontweight="bold", fontsize=9)

save_fig("SUMMARY_DASHBOARD.png", tight=False)




12) SUMMARY DASHBOARD
  [saved] SUMMARY_DASHBOARD.png


## 💾 13 — Metadata Export & Final Summary

In [16]:
print(baseline_df.columns.tolist())

['model', 'threshold', 'precision', 'recall', 'f1', 'pr_auc', 'ap', 'roc_auc', 'tn', 'fp', 'fn', 'tp', 'specificity', 'fpr', 'train_time_s', 'infer_time_s', 'pr_auc_ci_lo', 'pr_auc_ci_hi', 'ap_ci_lo', 'ap_ci_hi', 'f1_ci_lo', 'f1_ci_hi']


In [17]:

# ============================================================
# OPERATIONAL ANALYSIS — Training & Inference Time
# ============================================================
print("\n" + "="*70)
print("OPERATIONAL ANALYSIS — Phase 1 (Kaggle CC)")
print("="*70)

if not baseline_df.empty:
    _time_cols = [c for c in ["model","train_time_s","infer_time_s"] if c in baseline_df.columns]
    if len(_time_cols) >= 2:
        _time_df = baseline_df[_time_cols].sort_values("train_time_s")
        print("\n  Model | Train (s) | Inference (s)")
        print("  " + "-"*50)
        for _, _row in _time_df.iterrows():
            _inf = f"{_row['infer_time_s']:.4f}" if "infer_time_s" in _time_df.columns else "N/A"
            print(f"  {_row['model']:<25} | {_row['train_time_s']:>9.3f} | {_inf:>14}")

        _fig, _axes = plt.subplots(1, 2, figsize=(14, 5))
        _fig.suptitle("Operational Efficiency — Phase 1 (Kaggle CC)",
                      fontsize=14, fontweight="bold")
        _models_ord  = _time_df["model"].tolist()
        _colors_ord  = [COLORS[m] for m in _models_ord]
        _axes[0].barh(_models_ord, _time_df["train_time_s"],
                      color=_colors_ord, edgecolor="black", linewidth=0.7)
        _axes[0].set_xlabel("Training Time (seconds)")
        _axes[0].set_title("Training Time by Model")
        _axes[0].invert_yaxis()
        if "infer_time_s" in _time_df.columns:
            _axes[1].barh(_models_ord, _time_df["infer_time_s"],
                          color=_colors_ord, edgecolor="black", linewidth=0.7)
            _axes[1].set_xlabel("Inference Time (seconds)")
            _axes[1].set_title("Inference Time by Model")
            _axes[1].invert_yaxis()
        plt.tight_layout()
        save_fig("operational_01_timing_comparison.png")

        _fastest = _time_df.iloc[0]["model"]
        _slowest = _time_df.iloc[-1]["model"]
        _ratio   = _time_df.iloc[-1]["train_time_s"] / max(_time_df.iloc[0]["train_time_s"], 1e-6)
        print(f"\n  Fastest to train : {_fastest}")
        print(f"  Slowest to train : {_slowest} ({_ratio:.1f}x slower)")

# ── RQ Mapping ────────────────────────────────────────────────
print("\n" + "="*70)
print("RQ MAPPING — Phase 1")
print("="*70)
print("""
RQ1 — How do gradient boosting models compare to classical baselines?
  Phase 1 answer: see baseline_df ranked by pr_auc above.
  Note: XGBoost/LightGBM typically outperform LR/RF on PR-AUC at 0.17%
  fraud rate, but gap significance requires N_REPEATS ≥ 10 for conclusions.

RQ2 — What data characteristics impact model performance?
  Phase 1 answer: sensitivity analyses (Sections 6 & 7) show that:
  — PR-AUC degrades sharply below 0.5% fraud prevalence for all models.
  — Simpler models lose stability faster at very small training sizes.
  Full RQ2 requires cross-dataset comparison (addressed in Phase 3, Sec 13).""")

# ============================================================
# 12) METADATA EXPORT
# ============================================================
meta = {
    "dataset": str(DATA_PATH),
    "n_rows": int(len(df)),
    "fraud_prevalence": float(df[TARGET_COL].mean()),
    "features_used": FEATURES,
    "splits": {"train": TRAIN_SIZE, "val": VAL_SIZE, "test": TEST_SIZE},
    "models_tested": MODEL_NAMES,
    "imbalance_strategy": "class_weight=balanced (no SMOTE/resampling)",
    "sensitivity": {
        "imbalance_ratios": IMB_RATIOS,
        "size_fractions": SIZE_FRACS,
        "repeats": N_REPEATS,
        "bootstrap_samples": N_BOOTSTRAP,
    },
    "best_model_by_pr_auc": str(baseline_df.iloc[0]["model"]),
    "best_pr_auc": float(baseline_df.iloc[0]["pr_auc"]),
    "calibration": {
        m: {"brier": v["brier"], "ece": v["ece"]}
        for m, v in calib_stats.items()
    },
}
with open(RES_DIR / "phase1_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

# ============================================================
# DONE
# ============================================================
print("\n" + "="*70)
print("PHASE 1 COMPLETE ✅")
print("="*70)
figs = sorted(FIG_DIR.glob("*.png"))
print(f"\n{len(figs)} figures saved to: {FIG_DIR}")
for p in figs:
    print(f"  {p.name}")
print(f"\nResults saved to: {RES_DIR}")
print(f"\nNew outputs in this corrected version:")
print(f"  calibration_01_reliability_diagram.png")
print(f"  calibration_02_scores.png")
print(f"  significance_01_pairwise_heatmap.png")
print(f"  phase1_calibration_stats.csv")
print(f"  phase1_significance_tests.csv")

# ── Phase 1 Final Summary ─────────────────────────────────────────
print("\n" + "="*70)
print("PHASE 1 — FINAL RESULTS SUMMARY")
print("="*70)

# Main results table
summary_rows = []
for _, row in baseline_df.iterrows():
    rob_row = rob_summary[rob_summary['model']==row['model']]
    cal_row = calib_df[calib_df['Model']==row['model']] if 'calib_df' in dir() else pd.DataFrame()
    summary_rows.append({
        "Model":        row['model'],
        "PR-AUC":       f"{row['pr_auc']:.4f} [{row['pr_auc_ci_lo']:.4f}–{row['pr_auc_ci_hi']:.4f}]",
        "AP":           f"{row['ap']:.4f} [{row['ap_ci_lo']:.4f}–{row['ap_ci_hi']:.4f}]",
        "F1":           f"{row['f1']:.4f} [{row['f1_ci_lo']:.4f}–{row['f1_ci_hi']:.4f}]",
        "Recall":       f"{row['recall']:.4f}",
        "Precision":    f"{row['precision']:.4f}",
        "Train(s)":     f"{row['train_time_s']:.2f}",
        "Brier":        f"{calib_stats[row['model']]['brier']:.4f}" if row['model'] in calib_stats else "–",
        "ECE":          f"{calib_stats[row['model']]['ece']:.4f}" if row['model'] in calib_stats else "–",
    })

final_df = pd.DataFrame(summary_rows)
styled_table(final_df,
             higher_is_better=["PR-AUC","AP","F1","Recall","Precision"],
             title="Phase 1 — Complete Results Table (Test Set, 95% CI in brackets)")

print_summary("Phase 1 Complete", [
    ("Dataset", "Kaggle Credit Card Fraud — 283,726 transactions"),
    ("Best model overall", f"{baseline_df.iloc[0]['model']} (PR-AUC={baseline_df.iloc[0]['pr_auc']:.4f})"),
    ("Best calibration", f"{calib_df.iloc[0]['Model']} (ECE={calib_df.iloc[0]['ECE']:.4f})"),
    ("Most stable", f"{rob_summary.loc[rob_summary['pr_auc_std'].idxmin(),'model']} (std={rob_summary['pr_auc_std'].min():.4f})"),
    ("Fastest", f"{baseline_df.loc[baseline_df['train_time_s'].idxmin(),'model']} ({baseline_df['train_time_s'].min():.2f}s)"),
    ("Outputs saved to", str(OUT_DIR)),
])



OPERATIONAL ANALYSIS — Phase 1 (Kaggle CC)

  Model | Train (s) | Inference (s)
  --------------------------------------------------
  LightGBM                  |     1.609 |         0.0599
  Logistic Regression       |     3.496 |         0.0190
  XGBoost                   |     3.689 |         0.0247
  Random Forest             |    11.895 |         0.0637
  [saved] operational_01_timing_comparison.png

  Fastest to train : LightGBM
  Slowest to train : Random Forest (7.4x slower)

RQ MAPPING — Phase 1

RQ1 — How do gradient boosting models compare to classical baselines?
  Phase 1 answer: see baseline_df ranked by pr_auc above.
  Note: XGBoost/LightGBM typically outperform LR/RF on PR-AUC at 0.17%
  fraud rate, but gap significance requires N_REPEATS ≥ 10 for conclusions.

RQ2 — What data characteristics impact model performance?
  Phase 1 answer: sensitivity analyses (Sections 6 & 7) show that:
  — PR-AUC degrades sharply below 0.5% fraud prevalence for all models.
  — Simpler mod

,Model,PR-AUC,AP,F1,Recall,Precision,Train(s),Brier,ECE
0,XGBoost,0.8088 [0.7120–0.8920],0.8092 [0.7135–0.8924],0.8397 [0.7711–0.8951],0.7746,0.9167,3.69,0.0010,0.0029
1,LightGBM,0.8044 [0.7184–0.8864],0.7989 [0.7134–0.8838],0.7805 [0.6964–0.8635],0.6761,0.9231,1.61,0.0007,0.0014
2,Random Forest,0.7519 [0.6337–0.8465],0.7531 [0.6383–0.8478],0.7846 [0.7033–0.8501],0.7183,0.8644,11.90,0.0012,0.0111
3,Logistic Regression,0.6566 [0.5460–0.7885],0.6342 [0.5205–0.7792],0.7939 [0.7178–0.8595],0.7324,0.8667,3.50,0.0228,0.0715



┌────────────────────────────────────────────────────────────┐
│  📊 Phase 1 Complete                                       │
├────────────────────────────────────────────────────────────┤
│  Dataset: Kaggle Credit Card Fraud — 283,726 transactions  │
│  Best model overall: XGBoost (PR-AUC=0.8088)               │
│  Best calibration: LightGBM (ECE=0.0014)                   │
│  Most stable: LightGBM (std=0.0250)                        │
│  Fastest: LightGBM (1.61s)                                 │
│  Outputs saved to: c:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Phase 1\output phase 1│
└────────────────────────────────────────────────────────────┘
